In [1]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt


In [2]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
# List of tickers to download
TICKERS = [
    "BTC-USD",
    "ETH-USD",
    "TQQQ",
    "UPRO",
    "SMH.L",
    "UCO",
    "AAPL",
    "GOOG",
    "GLD",
    "QQQ",
    "SPY",
]

START_DATE = '2018-01-01'

# Validate config shape (a bare string would iterate character-by-character)
if isinstance(TICKERS, str):
    raise ValueError("TICKERS must be a list of ticker symbols, not a single string.")
if not isinstance(TICKERS, (list, tuple)):
    raise ValueError(f"TICKERS must be a list of ticker symbols; got {type(TICKERS).__name__}.")

# Normalize once: strip, drop blanks, preserve order, remove duplicates
TICKERS = list(dict.fromkeys(str(t).strip() for t in TICKERS if str(t).strip()))
if not TICKERS:
    raise ValueError("TICKERS is empty. Provide at least one ticker symbol.")

START_DATE = str(START_DATE).strip()
try:
    start_ts = pd.Timestamp(START_DATE)
except Exception as exc:
    raise ValueError(f"START_DATE={START_DATE!r} is not a valid date.") from exc
if start_ts.normalize() > pd.Timestamp.today().normalize():
    raise ValueError(f"START_DATE={START_DATE!r} is in the future.")

# Download data from start date onwards (explicit column grouping for stable MultiIndex)
try:
    stock_data = yf.download(
        TICKERS,
        start=START_DATE,
        interval="1d",
        group_by="column",
        threads=False,
    )
except Exception as exc:
    raise RuntimeError(f"yfinance download failed for TICKERS={TICKERS!r}: {exc}") from exc

if stock_data is None or stock_data.empty:
    raise ValueError(f"Failed to download data for TICKERS={TICKERS!r} from yfinance")

def _select_close(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ("Close", ticker) in df.columns:
            return df[("Close", ticker)]
        if (ticker, "Close") in df.columns:
            return df[(ticker, "Close")]
        return None
    if len(TICKERS) == 1 and "Close" in df.columns:
        return df["Close"]
    return None

failed_tickers = []
for t in list(TICKERS):
    close_s = _select_close(stock_data, t)
    if close_s is None or close_s.dropna().empty:
        failed_tickers.append(t)

if failed_tickers:
    print(
        f"WARNING: dropping {len(failed_tickers)} ticker(s) with no usable Close data: "
        f"{failed_tickers}"
    )
    TICKERS = [t for t in TICKERS if t not in set(failed_tickers)]

if not TICKERS:
    raise ValueError(
        "No usable Close data for any requested ticker after download. "
        f"Failed: {failed_tickers!r}."
    )

# Bootstrap only: some helper cells still read a global TICKER name.
# Analysis loops every symbol in TICKERS (not just this one).
TICKER = TICKERS[0]

print(f"Successfully downloaded {len(stock_data)} records for {len(TICKERS)} tickers from {START_DATE}")
print(f"Tickers: {', '.join(TICKERS)}")
print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
print("\nFirst 5 rows:")
print(stock_data.head())

# Display the downloaded data
stock_data


[                       0%                       ]

[*********             18%                       ]  2 of 11 completed

[*************         27%                       ]  3 of 11 completed

[*****************     36%                       ]  4 of 11 completed

[**********************45%                       ]  5 of 11 completed

...

[**********************91%*******************    ]  10 of 11 completed

[*********************100%***********************]  11 of 11 completed

[*********************100%***********************]  11 of 11 completed

Successfully downloaded 3144 records for 11 tickers from 2018-01-01
Tickers: BTC-USD, ETH-USD, TQQQ, UPRO, SMH.L, UCO, AAPL, GOOG, GLD, QQQ, SPY
Data range: 2018-01-01 to 2026-08-10

First 5 rows:
Price           Close                                                   \
Ticker           AAPL       BTC-USD     ETH-USD         GLD       GOOG   
Date                                                                     
2018-01-01        NaN  13657.200195  772.640991         NaN        NaN   
2018-01-02  40.267086  14982.099609  884.443970  125.150002  52.784615   
2018-01-03  40.260059  15201.000000  962.719971  124.820000  53.650974   
2018-01-04  40.447063  15599.200195  980.921997  125.459999  53.845268   
2018-01-05  40.907581  17429.500000  997.719971  125.330002  54.629845   

Price                                                         ...  \
Ticker             QQQ SMH.L         SPY      TQQQ       UCO  ...   
Date                                                          ...   
201

Price            Close                                                     \
Ticker            AAPL       BTC-USD      ETH-USD         GLD        GOOG   
Date                                                                        
2018-01-01         NaN  13657.200195   772.640991         NaN         NaN   
2018-01-02   40.267086  14982.099609   884.443970  125.150002   52.784615   
2018-01-03   40.260059  15201.000000   962.719971  124.820000   53.650974   
2018-01-04   40.447063  15599.200195   980.921997  125.459999   53.845268   
2018-01-05   40.907581  17429.500000   997.719971  125.330002   54.629845   
...                ...           ...          ...         ...         ...   
2026-08-06  312.410004  64262.113281  1902.057495  389.670013  356.619995   
2026-08-07  313.329987  64880.191406  1913.279419  398.470001  353.470001   
2026-08-08         NaN  64904.687500  1915.532959         NaN         NaN   
2026-08-09         NaN  64844.886719  1908.682739         NaN         NaN   
2026-08-10         NaN  65275.160156  1927.099976         NaN         NaN   

Price                                                                  ...  \
Ticker             QQQ       SMH.L         SPY       TQQQ         UCO  ...   
Date                                                                   ...   
2018-01-01         NaN         NaN         NaN        NaN         NaN  ...   
2018-01-02  150.057205         NaN  235.954239   5.778714  147.500000  ...   
2018-01-03  151.515320         NaN  237.446671   5.947906  153.812500  ...   
2018-01-04  151.780365         NaN  238.447495   5.982777  154.625000  ...   
2018-01-05  153.304703         NaN  240.036545   6.163460  153.125000  ...   
...                ...         ...         ...        ...         ...  ...   
2026-08-06  714.650024  108.199997  768.559998  72.029999   37.169998  ...   
2026-08-07  723.030029  108.540001  773.260010  74.470001   37.000000  ...   
2026-08-08         NaN         NaN         NaN        NaN         NaN  ...   
2026-08-09         NaN         NaN         NaN        NaN         NaN  ...   
2026-08-10         NaN  109.820000         NaN        NaN         NaN  ...   

Price            Volume                                                  \
Ticker          BTC-USD     ETH-USD         GLD        GOOG         QQQ   
Date                                                                      
2018-01-01  10291200000  2595760128         NaN         NaN         NaN   
2018-01-02  16846600192  5783349760  11762500.0  24752000.0  32573300.0   
2018-01-03  16871900160  5093159936   7904300.0  28604000.0  29383600.0   
2018-01-04  21783199744  6502859776   7329700.0  20092000.0  24776100.0   
2018-01-05  23840899072  6683149824   5739900.0  25582000.0  26992300.0   
...                 ...         ...         ...         ...         ...   
2026-08-06  18529402711  7974662808  10954300.0  16513400.0  33035900.0   
2026-08-07  22165720102  9081145963  13301600.0  14480400.0  31024100.0   
2026-08-08  12350094271  3746718589         NaN         NaN         NaN   
2026-08-09  13234538380  4645577105         NaN         NaN         NaN   
2026-08-10  14768925696  5978470400         NaN         NaN         NaN   

Price                                                               
Ticker         SMH.L         SPY        TQQQ        UCO       UPRO  
Date                                                                
2018-01-01       NaN         NaN         NaN        NaN        NaN  
2018-01-02       NaN  86655700.0  91735200.0   322528.0  8092800.0  
2018-01-03       NaN  90070400.0  85224000.0   561488.0  6100800.0  
2018-01-04       NaN  80636400.0  70024800.0   481936.0  7736400.0  
2018-01-05       NaN  83524000.0  82380000.0   331360.0  8523000.0  
...              ...         ...         ...        ...        ...  
2026-08-06  276537.0  38416900.0  60887200.0  3045300.0  1637100.0  
2026-08-07  318767.0  43557000.0  59134100.0  2466200.0  1832000.0  
2026-08-08       N

In [3]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

# Make sure stock_data / TICKERS are available from the download cell
if "stock_data" not in globals() or stock_data is None or getattr(stock_data, "empty", True):
    raise ValueError("stock_data is missing or empty. Run the download cell first.")
if "TICKERS" not in globals() or not TICKERS:
    raise ValueError("TICKERS is missing or empty. Run the download cell first.")

indicators_by_ticker = {}
_indicator_failures = []

# Helper indicator implementations (shared across tickers)
def validate_kama_params(period, fast_ema_constant, slow_ema_constant):
    try:
        period = int(period)
        fast_ema_constant = int(fast_ema_constant)
        slow_ema_constant = int(slow_ema_constant)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "KAMA parameters must be integers; "
            f"got period={period!r}, fast={fast_ema_constant!r}, slow={slow_ema_constant!r}"
        ) from exc
    if period <= 0 or fast_ema_constant <= 0 or slow_ema_constant <= 0:
        raise ValueError("KAMA parameters must be positive integers.")
    if fast_ema_constant >= slow_ema_constant:
        raise ValueError("KAMA fastest EMA constant must be less than slowest EMA constant.")
    return period, fast_ema_constant, slow_ema_constant

def _compute_kama_core(close, period, fast_ema_constant, slow_ema_constant):
    if close.dropna().shape[0] <= period:
        raise ValueError(f"KAMA requires more than {period} rows of close data")

    change = (close - close.shift(period)).abs()
    volatility = close.diff().abs().rolling(window=period).sum()
    er = change / (volatility + 1e-9)
    sc = (
        (er * (2 / (fast_ema_constant + 1) - 2 / (slow_ema_constant + 1)))
        + 2 / (slow_ema_constant + 1)
    ) ** 2
    sc = sc.clip(0, 1).fillna(0)

    kama_values = np.full(len(close), np.nan)
    seed = close.iloc[period]
    if pd.isna(seed):
        return pd.Series(kama_values, index=close.index)

    kama_values[period] = seed
    sc_values = sc.to_numpy(dtype=float)
    close_values = close.to_numpy(dtype=float)
    for i in range(period + 1, len(close)):
        if np.isnan(close_values[i]):
            kama_values[i] = np.nan
            continue
        prev = kama_values[i - 1]
        if np.isnan(prev):
            kama_values[i] = np.nan
            continue
        kama_values[i] = prev + sc_values[i] * (close_values[i] - prev)

    kama_values[:period] = np.nan
    return pd.Series(kama_values, index=close.index)

def compute_kama_series(close, period, fast_ema_constant, slow_ema_constant, index=None):
    period, fast_ema_constant, slow_ema_constant = validate_kama_params(
        period,
        fast_ema_constant,
        slow_ema_constant,
    )
    if not isinstance(close, pd.Series):
        if index is None:
            raise ValueError("index is required when close is not a Series")
        close = pd.Series(close, index=index)
    close = close.astype(float)
    return _compute_kama_core(close, period, fast_ema_constant, slow_ema_constant)

def validate_supertrend_params(atr_length, factor):
    try:
        atr_length_num = float(atr_length)
        factor_num = float(factor)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "SUPERTREND parameters must be numeric; "
            f"got atr_length={atr_length!r}, factor={factor!r}"
        ) from exc
    if not np.isfinite(atr_length_num) or not np.isfinite(factor_num):
        raise ValueError("SUPERTREND ATR length and factor must be finite numbers.")
    if atr_length_num != int(atr_length_num):
        raise ValueError(
            f"SUPERTREND ATR length must be an integer; got atr_length={atr_length!r}"
        )
    atr_length = int(atr_length_num)
    # Reject silent float truncation (e.g. 2.7 -> 2) while still allowing 3.0
    if factor_num != int(factor_num):
        raise ValueError(
            "SUPERTREND factor must be a whole number for this grid "
            f"(sensitivity uses int grids); got factor={factor!r}"
        )
    factor = int(factor_num)
    if atr_length <= 0 or factor <= 0:
        raise ValueError("SUPERTREND ATR length and factor must be positive integers.")
    return atr_length, factor

def _compute_supertrend_core(high, low, close, atr_length, factor):
    if len(high) != len(close) or len(low) != len(close):
        raise ValueError("High, Low, and Close arrays must have the same length for SUPERTREND")
    if len(close) == 0:
        raise ValueError("SUPERTREND requires a non-empty OHLC series")
    if len(close) <= atr_length:
        raise ValueError(f"SUPERTREND requires more than {atr_length} rows of OHLC data")

    high_arr = np.asarray(high, dtype=float)
    low_arr = np.asarray(low, dtype=float)
    close_arr = np.asarray(close, dtype=float)

    finite_mask = np.isfinite(high_arr) & np.isfinite(low_arr) & np.isfinite(close_arr)
    if not finite_mask.any():
        raise ValueError("SUPERTREND requires at least one finite OHLC row")
    if np.any(high_arr[finite_mask] < low_arr[finite_mask]):
        raise ValueError("SUPERTREND requires High >= Low on all finite rows")

    atr = talib.ATR(high_arr, low_arr, close_arr, timeperiod=atr_length)
    hl2 = (high_arr + low_arr) / 2.0
    basic_upper = hl2 + factor * atr
    basic_lower = hl2 - factor * atr

    final_upper = np.full(len(close_arr), np.nan)
    final_lower = np.full(len(close_arr), np.nan)
    supertrend = np.full(len(close_arr), np.nan)
    direction = np.full(len(close_arr), np.nan)

    for i in range(len(close_arr)):
        # Gap / bad bar: carry prior state forward so trend memory is not reset
        if (
            np.isnan(atr[i])
            or not np.isfinite(close_arr[i])
            or not np.isfinite(high_arr[i])
            or not np.isfinite(low_arr[i])
        ):
            if i > 0:
                final_upper[i] = final_upper[i - 1]
                final_lower[i] = final_lower[i - 1]
                direction[i] = direction[i - 1]
                supertrend[i] = supertrend[i - 1]
            continue

        if i == 0 or np.isnan(final_lower[i - 1]):
            final_lower[i] = basic_lower[i]
        elif basic_lower[i] > final_lower[i - 1] or close_arr[i - 1] < final_lower[i - 1]:
            final_lower[i] = basic_lower[i]
        else:
            final_lower[i] = final_lower[i - 1]

        if i == 0 or np.isnan(final_upper[i - 1]):
            final_upper[i] = basic_upper[i]
        elif basic_upper[i] < final_upper[i - 1] or close_arr[i - 1] > final_upper[i - 1]:
            final_upper[i] = basic_upper[i]
        else:
            final_upper[i] = final_upper[i - 1]

        # Seed first direction from bands, then prior-close momentum (avoid hl2 tie -> always +1)
        if i == 0 or np.isnan(direction[i - 1]):
            if close_arr[i] > final_upper[i]:
                direction[i] = 1.0
            elif close_arr[i] < final_lower[i]:
                direction[i] = -1.0
            elif i > 0 and np.isfinite(close_arr[i - 1]):
                direction[i] = 1.0 if close_arr[i] >= close_arr[i - 1] else -1.0
            else:
                direction[i] = 1.0 if close_arr[i] > hl2[i] else -1.0
        elif direction[i - 1] == 1.0:
            direction[i] = -1.0 if close_arr[i] < final_lower[i] else 1.0
        else:
            direction[i] = 1.0 if close_arr[i] > final_upper[i] else -1.0

        supertrend[i] = final_lower[i] if direction[i] == 1.0 else final_upper[i]

    return supertrend, direction

def compute_supertrend_series(high, low, close, atr_length, factor, index=None):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("SUPERTREND index length must match close length")
    supertrend, direction = _compute_supertrend_core(high, low, close, atr_length, factor)
    return (
        pd.Series(supertrend, index=index),
        pd.Series(direction, index=index),
    )

def validate_kalman_params(process_noise, measurement_noise):
    try:
        process_noise_num = float(process_noise)
        measurement_noise_num = float(measurement_noise)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "KALMAN parameters must be numeric; "
            f"got process_noise={process_noise!r}, measurement_noise={measurement_noise!r}"
        ) from exc
    if not np.isfinite(process_noise_num) or not np.isfinite(measurement_noise_num):
        raise ValueError("KALMAN process noise (Q) and measurement noise (R) must be finite numbers.")
    if process_noise_num != int(process_noise_num):
        raise ValueError(
            "KALMAN process noise (Q) must be a whole number for this grid "
            f"(sensitivity uses int grids); got process_noise={process_noise!r}"
        )
    if measurement_noise_num != int(measurement_noise_num):
        raise ValueError(
            "KALMAN measurement noise (R) must be a whole number for this grid "
            f"(sensitivity uses int grids); got measurement_noise={measurement_noise!r}"
        )
    process_noise = int(process_noise_num)
    measurement_noise = int(measurement_noise_num)
    if process_noise <= 0 or measurement_noise <= 0:
        raise ValueError("KALMAN process noise (Q) and measurement noise (R) must be positive integers.")
    return process_noise, measurement_noise

def _compute_kalman_core(close, process_noise, measurement_noise):
    close_arr = np.asarray(close, dtype=float)
    if close_arr.size == 0:
        raise ValueError("KALMAN requires a non-empty close series")
    if not np.isfinite(close_arr).any():
        raise ValueError("KALMAN requires at least one finite close price")

    kalman = np.full(close_arr.size, np.nan)
    q = float(process_noise)
    r = float(measurement_noise)

    first_idx = int(np.flatnonzero(np.isfinite(close_arr))[0])
    x = close_arr[first_idx]
    # Seed uncertainty from R so the first updates are not artificially overconfident
    p = float(measurement_noise)
    kalman[first_idx] = x

    for i in range(first_idx + 1, close_arr.size):
        z = close_arr[i]
        # Predict every bar (including gaps) so multi-bar holes inflate uncertainty
        p = p + q
        if not np.isfinite(z):
            # Carry estimate forward across gaps; do not update with bad measurements
            kalman[i] = x
            continue
        # Update: higher R trusts the filter prediction over noisy measurements
        k = p / (p + r)
        x = x + k * (z - x)
        p = (1.0 - k) * p
        kalman[i] = x

    return kalman

def compute_kalman_series(close, process_noise, measurement_noise, index=None):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("KALMAN index length must match close length")
    kalman = _compute_kalman_core(close, process_noise, measurement_noise)
    return pd.Series(kalman, index=index)

def validate_alma_params(length, offset, sigma):
    try:
        length_num = float(length)
        offset_num = float(offset)
        sigma_num = float(sigma)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "ALMA parameters must be numeric; "
            f"got length={length!r}, offset={offset!r}, sigma={sigma!r}"
        ) from exc
    if not np.isfinite(length_num) or not np.isfinite(offset_num) or not np.isfinite(sigma_num):
        raise ValueError("ALMA length, offset, and sigma must be finite numbers.")
    if length_num != int(length_num):
        raise ValueError(
            "ALMA length must be a whole number for this grid "
            f"(sensitivity uses int grids); got length={length!r}"
        )
    if offset_num != int(offset_num):
        raise ValueError(
            "ALMA offset must be a whole number for this grid "
            f"(sensitivity uses int grids; offset is hundredths, e.g. 85 = 0.85); "
            f"got offset={offset!r}"
        )
    if sigma_num != int(sigma_num):
        raise ValueError(
            "ALMA sigma must be a whole number for this grid "
            f"(sensitivity uses int grids); got sigma={sigma!r}"
        )
    length = int(length_num)
    offset = int(offset_num)
    sigma = int(sigma_num)
    if length <= 1:
        raise ValueError("ALMA length must be an integer greater than 1.")
    # 0..100 hundredths maps to classic ALMA distribution offset [0.0, 1.0]
    if offset < 0 or offset > 100:
        raise ValueError("ALMA offset must be an integer in 0..100 (hundredths of the distribution offset).")
    if sigma <= 0:
        raise ValueError("ALMA sigma must be a positive integer.")
    return length, offset, sigma

def _compute_alma_core(close, length, offset, sigma):
    # Defensive guards so direct callers cannot hit ZeroDivisionError / bad shapes.
    if length <= 1:
        raise ValueError("ALMA length must be an integer greater than 1.")
    if sigma <= 0:
        raise ValueError("ALMA sigma must be a positive integer.")
    if offset < 0 or offset > 100:
        raise ValueError("ALMA offset must be an integer in 0..100 (hundredths of the distribution offset).")

    close_arr = np.asarray(close, dtype=float)
    if close_arr.size == 0:
        raise ValueError("ALMA requires a non-empty close series")
    if close_arr.size < length:
        raise ValueError(f"ALMA requires at least {length} rows of close data")
    if not np.isfinite(close_arr).any():
        raise ValueError("ALMA requires at least one finite close price")

    offset_f = offset / 100.0
    m = int(np.floor(offset_f * (length - 1)))
    s = length / float(sigma)
    idxs = np.arange(length, dtype=float)
    weights = np.exp(-((idxs - m) ** 2) / (2.0 * s * s))
    norm = weights.sum()
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError("ALMA weights are degenerate for the selected parameters")

    # Vectorized weighted window (avoids Python per-bar loop on large grids).
    alma = np.full(close_arr.size, np.nan)
    windows = np.lib.stride_tricks.sliding_window_view(close_arr, window_shape=length)
    finite_mask = np.isfinite(windows).all(axis=1)
    if finite_mask.any():
        alma_vals = np.full(windows.shape[0], np.nan)
        alma_vals[finite_mask] = windows[finite_mask] @ weights / norm
        alma[length - 1:] = alma_vals

    return alma

def compute_alma_series(close, length, offset, sigma, index=None):
    length, offset, sigma = validate_alma_params(length, offset, sigma)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("ALMA index length must match close length")
    alma = _compute_alma_core(close, length, offset, sigma)
    return pd.Series(alma, index=index)

def _ohlcv_col(frame, field, ticker):
    if isinstance(frame.columns, pd.MultiIndex):
        if (field, ticker) in frame.columns:
            return frame[(field, ticker)].values
        if (ticker, field) in frame.columns:
            return frame[(ticker, field)].values
        raise KeyError(f"{field} not found for ticker {ticker!r} in stock_data columns")
    if field not in frame.columns:
        raise KeyError(f"{field} not found in stock_data columns")
    return frame[field].values

# Shared indicator defaults (constant across tickers)
STC_EMA_SHORT = 23
STC_EMA_LONG = 50
STC_CYCLE_PERIOD = 10
AROON_PERIOD = 14

for TICKER in TICKERS:
    try:
        # Extract OHLCV data (handling multi-level columns from yfinance)
        if (not isinstance(stock_data.columns, pd.MultiIndex)) and len(TICKERS) > 1:
            raise ValueError(
                "stock_data has flat columns but multiple TICKERS were requested; "
                "expected a MultiIndex from yfinance."
            )
        close = _ohlcv_col(stock_data, "Close", TICKER)
        high = _ohlcv_col(stock_data, "High", TICKER)
        low = _ohlcv_col(stock_data, "Low", TICKER)
        open_ = _ohlcv_col(stock_data, "Open", TICKER)
        volume = _ohlcv_col(stock_data, "Volume", TICKER)

        print(f"Calculating technical indicators for {TICKER}...")

        # Simple Moving Averages
        sma_20 = talib.SMA(close, timeperiod=20)
        sma_50 = talib.SMA(close, timeperiod=50)

        # Exponential Moving Averages
        ema_12 = talib.EMA(close, timeperiod=12)
        ema_26 = talib.EMA(close, timeperiod=26)

        # MACD
        macd, macdsignal, macdhist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

        # RSI
        rsi = talib.RSI(close, timeperiod=14)

        # Stochastic RSI
        stochrsi_k, stochrsi_d = talib.STOCHRSI(close, timeperiod=14, fastk_period=3, fastd_period=3, fastd_matype=0)

        # VWAP (manual calculation)
        typical_price = (high + low + close) / 3
        price_volume = typical_price * volume
        cumulative_price_volume = np.cumsum(price_volume)
        cumulative_volume = np.cumsum(volume)
        vwap = cumulative_price_volume / cumulative_volume

        # Schaff Trend Cycle (defaults: EMA 23/50, stochastic cycle 10)

        stc_ema_short = talib.EMA(close, timeperiod=STC_EMA_SHORT)
        stc_ema_long = talib.EMA(close, timeperiod=STC_EMA_LONG)
        stc_macd = stc_ema_short - stc_ema_long

        stc_macd_highest = talib.MAX(stc_macd, timeperiod=STC_CYCLE_PERIOD)
        stc_macd_lowest = talib.MIN(stc_macd, timeperiod=STC_CYCLE_PERIOD)
        stc_macd_denom = stc_macd_highest - stc_macd_lowest
        with np.errstate(divide="ignore", invalid="ignore"):
            stc_k_macd = np.where(np.abs(stc_macd_denom) > 0, 100.0 * (stc_macd - stc_macd_lowest) / stc_macd_denom, 0.0)
        stc_d_macd = talib.EMA(stc_k_macd, timeperiod=STC_CYCLE_PERIOD)

        stc_d_highest = talib.MAX(stc_d_macd, timeperiod=STC_CYCLE_PERIOD)
        stc_d_lowest = talib.MIN(stc_d_macd, timeperiod=STC_CYCLE_PERIOD)
        stc_d_denom = stc_d_highest - stc_d_lowest
        with np.errstate(divide="ignore", invalid="ignore"):
            stc_k = np.where(np.abs(stc_d_denom) > 0, 100.0 * (stc_d_macd - stc_d_lowest) / stc_d_denom, 0.0)
        stc_d = talib.EMA(stc_k, timeperiod=STC_CYCLE_PERIOD)

        # AROON
        if len(high) != len(close) or len(low) != len(close):
            raise ValueError("High, Low, and Close arrays must have the same length for AROON")
        if len(close) <= AROON_PERIOD:
            raise ValueError(f"AROON requires more than {AROON_PERIOD} rows of OHLC data")
        aroon_down, aroon_up = talib.AROON(
            high.astype(float, copy=False),
            low.astype(float, copy=False),
            timeperiod=AROON_PERIOD,
        )

        # KAMA (Kaufman Adaptive Moving Average — custom implementation for full parameter control)
        KAMA_PERIOD = 10
        KAMA_FAST_EMA_CONSTANT = 2
        KAMA_SLOW_EMA_CONSTANT = 30

        kama = compute_kama_series(
            close,
            KAMA_PERIOD,
            KAMA_FAST_EMA_CONSTANT,
            KAMA_SLOW_EMA_CONSTANT,
            index=stock_data.index,
        ).values

        # SUPERTREND (ATR from TA-Lib + SuperTrend band/direction logic)
        SUPERTREND_ATR_LENGTH = 10
        SUPERTREND_FACTOR = 3

        supertrend, supertrend_direction = compute_supertrend_series(
            high,
            low,
            close,
            SUPERTREND_ATR_LENGTH,
            SUPERTREND_FACTOR,
            index=stock_data.index,
        )
        supertrend = supertrend.values
        supertrend_direction = supertrend_direction.values

        # KALMAN (1D Kalman filter — custom implementation; Q=process noise, R=measurement noise)
        KALMAN_PROCESS_NOISE = 1
        KALMAN_MEASUREMENT_NOISE = 10

        kalman = compute_kalman_series(
            close,
            KALMAN_PROCESS_NOISE,
            KALMAN_MEASUREMENT_NOISE,
            index=stock_data.index,
        ).values

        # ALMA (Arnaud Legoux Moving Average — custom implementation; TA-Lib has no ALMA)
        # Offset is stored as hundredths (85 = 0.85) so sensitivity int-grids stay coherent.
        ALMA_LENGTH = 9
        ALMA_OFFSET = 85
        ALMA_SIGMA = 6

        alma = compute_alma_series(
            close,
            ALMA_LENGTH,
            ALMA_OFFSET,
            ALMA_SIGMA,
            index=stock_data.index,
        ).values

        # Create indicators dataframe
        indicators_df = pd.DataFrame({
            "Date": stock_data.index,
            "Close": close,
            "SMA_20": sma_20,
            "SMA_50": sma_50,
            "EMA_12": ema_12,
            "EMA_26": ema_26,
            "MACD": macd,
            "MACD_Signal": macdsignal,
            "MACD_Hist": macdhist,
            "RSI": rsi,
            "StochRSI_K": stochrsi_k,
            "StochRSI_D": stochrsi_d,
            "VWAP": vwap,
            "STC_K": stc_k,
            "STC_D": stc_d,
            "AROON_Down": aroon_down,
            "AROON_Up": aroon_up,
            "KAMA": kama,
            "SUPERTREND": supertrend,
            "SUPERTREND_Direction": supertrend_direction,
            "KALMAN": kalman,
            "ALMA": alma,
        })

        indicators_by_ticker[TICKER] = indicators_df
        print(f"All technical indicators calculated for {TICKER}!")
        print(f"Data shape: {indicators_df.shape}")
    except Exception as exc:
        _indicator_failures.append(f"{TICKER}: {type(exc).__name__}: {exc}")
        print(f"SKIP indicators for {TICKER}: {type(exc).__name__}: {exc}")
        continue

if _indicator_failures:
    print(f"Indicator failures: {len(_indicator_failures)}")
    for _detail in _indicator_failures:
        print(f"  - {_detail}")

if not indicators_by_ticker:
    raise ValueError(
        f"Technical indicators failed for every ticker in TICKERS={TICKERS!r}."
    )

_ok = ", ".join(indicators_by_ticker)
print(f"Calculated indicators for {len(indicators_by_ticker)} tickers: {_ok}")
# Keep indicators_df as the last successful ticker's frame for any single-frame refs
TICKER = next(reversed(indicators_by_ticker))
indicators_df = indicators_by_ticker[TICKER]
indicators_df.tail(5)


Calculating technical indicators for BTC-USD...
All technical indicators calculated for BTC-USD!
Data shape: (3144, 22)
Calculating technical indicators for ETH-USD...
All technical indicators calculated for ETH-USD!
Data shape: (3144, 22)
Calculating technical indicators for TQQQ...
All technical indicators calculated for TQQQ!
Data shape: (3144, 22)
Calculating technical indicators for UPRO...
All technical indicators calculated for UPRO!
Data shape: (3144, 22)
Calculating technical indicators for SMH.L...
All technical indicators calculated for SMH.L!
Data shape: (3144, 22)
Calculating technical indicators for UCO...
All technical indicators calculated for UCO!
Data shape: (3144, 22)
Calculating technical indicators for AAPL...
All technical indicators calculated for AAPL!
Data shape: (3144, 22)
Calculating technical indicators for GOOG...
All technical indicators calculated for GOOG!
Data shape: (3144, 22)
Calculating technical indicators for GLD...
All technical indicators calcula

All technical indicators calculated for QQQ!
Data shape: (3144, 22)
Calculating technical indicators for SPY...
All technical indicators calculated for SPY!
Data shape: (3144, 22)
Calculated indicators for 11 tickers: BTC-USD, ETH-USD, TQQQ, UPRO, SMH.L, UCO, AAPL, GOOG, GLD, QQQ, SPY


,Date,Close,SMA_20,SMA_50,EMA_12,EMA_26,MACD,MACD_Signal,MACD_Hist,RSI,...,VWAP,STC_K,STC_D,AROON_Down,AROON_Up,KAMA,SUPERTREND,SUPERTREND_Direction,KALMAN,ALMA
3139,2026-08-06,768.559998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,42.857143,92.857143,NaN,NaN,NaN,761.901174,NaN
3140,2026-08-07,773.260010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,35.714286,85.714286,NaN,NaN,NaN,765.046776,NaN
3141,2026-08-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,28.571429,78.571429,NaN,NaN,NaN,765.046776,NaN
3142,2026-08-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,21.428571,71.428571,NaN,NaN,NaN,765.046776,NaN
3143,2026-08-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,14.285714,64.285714,NaN,NaN,NaN,765.046776,NaN


In [4]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKERS already exist
if "stock_data" not in globals() or stock_data is None or getattr(stock_data, "empty", True):
    raise ValueError("stock_data is missing or empty. Run the download cell first.")
if "TICKERS" not in globals() or not TICKERS:
    raise ValueError("TICKERS is missing or empty. Run the download cell first.")

def select_close_series(df, ticker):
    ticker = str(ticker).strip()
    if not ticker:
        raise ValueError("ticker is empty when selecting Close series.")
    if isinstance(df.columns, pd.MultiIndex):
        if ("Close", ticker) in df.columns:
            s = df[("Close", ticker)]
        elif (ticker, "Close") in df.columns:
            s = df[(ticker, "Close")]
        else:
            raise KeyError(
                f"Close not found for ticker {ticker!r} in stock_data MultiIndex columns."
            )
    else:
        if len(TICKERS) > 1:
            raise ValueError(
                "stock_data has flat columns but multiple TICKERS were requested; "
                "expected a MultiIndex from yfinance."
            )
        if "Close" not in df.columns:
            raise KeyError("Close not found in stock_data columns.")
        s = df["Close"]
    return s.astype(float).squeeze()

TRAIN_RATIO = 0.60
if not isinstance(TRAIN_RATIO, (int, float)) or not (0.0 < float(TRAIN_RATIO) < 1.0):
    raise ValueError(
        f"TRAIN_RATIO must be a number strictly between 0 and 1; got {TRAIN_RATIO!r}."
    )
TRAIN_RATIO = float(TRAIN_RATIO)

close_by_ticker = {}
train_close_by_ticker = {}
val_close_by_ticker = {}
_bars_per_year_by_ticker = {}
_calendar_days_by_ticker = {}
_has_weekend_bars_by_ticker = {}
_prepare_failures = []

for TICKER in TICKERS:
    try:
        close = select_close_series(stock_data, TICKER)
        close = close.replace([np.inf, -np.inf], np.nan).dropna()
        if close.empty or len(close) < 2:
            raise ValueError(f"Close series for TICKER={TICKER!r} is missing or too short.")
        if not isinstance(close.index, pd.DatetimeIndex):
            close.index = pd.to_datetime(close.index, errors="coerce")
        if close.index.isna().any() or len(close.index) < 2:
            raise ValueError(
                f"Close index for TICKER={TICKER!r} must be datetime-like with >= 2 valid timestamps."
            )
        close = close.sort_index()
        if close.index.has_duplicates:
            close = close[~close.index.duplicated(keep="last")]
        if close.empty or len(close) < 2:
            raise ValueError(f"Close series for TICKER={TICKER!r} is missing or too short.")
        close.name = "price"

        split_idx = int(len(close) * TRAIN_RATIO)
        if split_idx < 1 or split_idx >= len(close):
            raise ValueError(
                f"TRAIN_RATIO={TRAIN_RATIO!r} produced an invalid split for TICKER={TICKER!r} "
                f"(len={len(close)}, split_idx={split_idx})."
            )
        train_close = close.iloc[:split_idx].copy()
        val_close = close.iloc[split_idx:].copy()
        if len(train_close) < 2 or len(val_close) < 2:
            raise ValueError(
                f"train/val split too short for TICKER={TICKER!r} "
                f"(train={len(train_close)}, val={len(val_close)})."
            )

        _calendar_days = max((close.index[-1] - close.index[0]).days, 1)
        _bars_per_year = len(close) / (_calendar_days / 365.25)
        if not np.isfinite(_bars_per_year) or _bars_per_year <= 0:
            raise ValueError(
                f"Could not infer bars/year for TICKER={TICKER!r} "
                f"(bars={len(close)}, calendar_days={_calendar_days})."
            )
        _bars_per_year_by_ticker[TICKER] = float(_bars_per_year)
        _calendar_days_by_ticker[TICKER] = int(_calendar_days)
        # Weekend prints are the primary 24/7 signal. bars/year > 330 is only trusted
        # on long enough samples (short weekday stocks can exceed 330 bars/year).
        _has_weekend_bars_by_ticker[TICKER] = bool(
            np.isin(close.index.dayofweek, [5, 6]).any()
        )

        close_by_ticker[TICKER] = close
        train_close_by_ticker[TICKER] = train_close
        val_close_by_ticker[TICKER] = val_close

        print(
            f"{TICKER}: train={train_close.index[0].date()} -> {train_close.index[-1].date()} | "
            f"val={val_close.index[0].date()} -> {val_close.index[-1].date()}"
        )
    except Exception as exc:
        _prepare_failures.append(f"{TICKER}: {type(exc).__name__}: {exc}")
        print(f"SKIP prepare for {TICKER}: {type(exc).__name__}: {exc}")
        continue

if _prepare_failures:
    print(f"Prepare failures: {len(_prepare_failures)}")
    for _detail in _prepare_failures:
        print(f"  - {_detail}")

if not close_by_ticker:
    raise ValueError(f"Price preparation failed for every ticker in TICKERS={TICKERS!r}.")

# Keep TICKERS aligned to symbols that actually prepared successfully
TICKERS = [t for t in TICKERS if t in close_by_ticker]

# Auto-detect market calendar per ticker (mixed crypto/stock universes supported)
_MIN_CALENDAR_DAYS_FOR_BARS_HEURISTIC = 120
_is_crypto_by_ticker = {
    t: bool(
        _has_weekend_bars_by_ticker[t]
        or (
            _calendar_days_by_ticker[t] >= _MIN_CALENDAR_DAYS_FOR_BARS_HEURISTIC
            and _bars_per_year_by_ticker[t] > 330
        )
    )
    for t in close_by_ticker
}
trading_days_per_year_by_ticker = {
    t: (365 if flag else 252) for t, flag in _is_crypto_by_ticker.items()
}

FREQ = "1D"


def _apply_market_calendar(ticker):
    """Set shared calendar globals from a prepared ticker's detected market type."""
    global IS_24_7_MARKET, TRADING_DAYS_PER_YEAR, YEAR_FREQ
    ticker = str(ticker).strip()
    if ticker not in trading_days_per_year_by_ticker:
        raise KeyError(
            f"No market calendar prepared for ticker {ticker!r}. "
            f"Known: {sorted(trading_days_per_year_by_ticker)}"
        )
    IS_24_7_MARKET = bool(_is_crypto_by_ticker[ticker])
    TRADING_DAYS_PER_YEAR = int(trading_days_per_year_by_ticker[ticker])
    if TRADING_DAYS_PER_YEAR <= 0:
        raise ValueError(
            f"TRADING_DAYS_PER_YEAR for {ticker!r} must be > 0; got {TRADING_DAYS_PER_YEAR!r}"
        )
    YEAR_FREQ = f"{TRADING_DAYS_PER_YEAR}D"


class _CalendarSyncDict(dict):
    """Dict that applies the accessed ticker's market calendar to shared globals."""

    def __getitem__(self, key):
        if not dict.__contains__(self, key):
            raise KeyError(key)
        _apply_market_calendar(key)
        return dict.__getitem__(self, key)

    def get(self, key, default=None):
        if dict.__contains__(self, key):
            return self[key]
        return default


close_by_ticker = _CalendarSyncDict(close_by_ticker)


def sample_years(price_series):
    if price_series is None or len(price_series) < 2:
        raise ValueError("sample_years requires a price series with at least 2 rows.")
    if TICKER in trading_days_per_year_by_ticker:
        tdpy = trading_days_per_year_by_ticker[TICKER]
    else:
        # Infer from the series itself when TICKER is unavailable/stale.
        idx = pd.DatetimeIndex(pd.to_datetime(price_series.index, errors="coerce"))
        if idx.isna().any():
            raise ValueError("sample_years requires a datetime-like price index.")
        _calendar_days = max((idx[-1] - idx[0]).days, 1)
        _bars_per_year = len(price_series) / (_calendar_days / 365.25)
        _weekend = bool(np.isin(idx.dayofweek, [5, 6]).any())
        _dense_long = (
            _calendar_days >= _MIN_CALENDAR_DAYS_FOR_BARS_HEURISTIC
            and np.isfinite(_bars_per_year)
            and _bars_per_year > 330
        )
        tdpy = 365 if (_weekend or _dense_long) else 252
    if not np.isfinite(tdpy) or tdpy <= 0:
        raise ValueError(f"sample_years got invalid trading-days-per-year={tdpy!r}")
    return max(len(price_series) / float(tdpy), 1e-9)


# Keep single-frame aliases pointed at the first prepared ticker for any legacy refs
TICKER = TICKERS[0]
close = close_by_ticker[TICKER]
train_close = train_close_by_ticker[TICKER]
val_close = val_close_by_ticker[TICKER]

_calendar_summary = ", ".join(
    f"{t}:{'crypto/365D' if _is_crypto_by_ticker[t] else 'stock/252D'}" for t in TICKERS
)
print(
    f"Prepared {len(close_by_ticker)} tickers | "
    f"FREQ='{FREQ}' | calendars=[{_calendar_summary}]"
)


BTC-USD: train=2018-01-01 -> 2023-03-01 | val=2023-03-02 -> 2026-08-10
ETH-USD: train=2018-01-01 -> 2023-03-01 | val=2023-03-02 -> 2026-08-10
TQQQ: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
UPRO: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
SMH.L: train=2020-12-01 -> 2024-05-01 | val=2024-05-02 -> 2026-08-10
UCO: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
AAPL: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
GOOG: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
GLD: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
QQQ: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
SPY: train=2018-01-02 -> 2023-02-24 | val=2023-02-27 -> 2026-08-07
Prepared 11 tickers | FREQ='1D' | calendars=[BTC-USD:crypto/365D, ETH-USD:crypto/365D, TQQQ:stock/252D, UPRO:stock/252D, SMH.L:stock/252D, UCO:stock/252D, AAPL:stock/252D, GOOG:stock/252D, GLD:stock/252D, QQQ:stock/252D, SPY:stock/252D]


In [5]:
# Define Indicator Grids

# Existing Triple EMA grid. MACD, AROON, STC, KAMA, SUPERTREND, KALMAN, RSI, ADX, DONCHIAN, TRIX, VORTEX, and ALMA are added below as separate grids; parameters are not mixed.
ema1_periods = list(range(4, 10, 1))    # Fast EMA (shortest-term)
ema2_periods = list(range(90, 110, 1))  # Medium EMA
ema3_periods = list(range(120, 140, 1)) # Slow EMA

# MACD periods for crossover strategy
macd_fast_periods = list(range(10, 50, 1))
macd_slow_periods = list(range(60, 100, 1))
macd_signal_periods = list(range(40, 120, 1))

# AROON timeperiods for Up/Down crossover strategy
aroon_timeperiods = list(range(4, 80, 1))
AROON_ENTRY_LEVEL = 49

# STC EMA and cycle periods for Schaff Trend Cycle strategy
stc_ema_short_periods = list(range(10, 50, 1))
stc_ema_long_periods = list(range(60, 100, 1))
stc_cycle_periods = list(range(20, 100, 1))
STC_BUY_LEVEL = 50
STC_SELL_LEVEL = 50

# KAMA periods and EMA constants for price crossover strategy
kama_periods = list(range(5, 30, 5))
kama_fast_ema_constants = list(range(2, 40, 1))
kama_slow_ema_constants = list(range(60, 100, 1))

# SUPERTREND ATR length and factor/multiplier for trend-flip strategy
supertrend_atr_lengths = list(range(4, 56, 1))
supertrend_factors = list(range(1, 30, 1))

# KALMAN process noise (Q) and measurement noise (R) for price vs filter strategy
kalman_process_noise_values = list(range(1, 5, 1))
kalman_measurement_noise_values = list(range(100, 160, 1))

# RSI timeperiods for midline crossover strategy
rsi_timeperiods = list(range(40, 80, 1))
RSI_CROSS_LEVEL = 50

# ADX DI length and smoothing timeperiods for +DI/-DI crossover with trend-strength filter
adx_di_lengths = list(range(4, 60, 1))
adx_smoothing_timeperiods = list(range(20, 80, 1))
ADX_TREND_LEVEL = 20

# DONCHIAN timeperiods for upper-band breakout / middle-band trailing-stop strategy
donchian_timeperiods = list(range(10, 60, 1))

# TRIX timeperiods for zero-line crossover strategy
trix_timeperiods = list(range(4, 80, 1))
TRIX_CROSS_LEVEL = 0

# VORTEX timeperiods for VI+/VI- crossover strategy
vortex_timeperiods = list(range(40, 140, 1))

# ALMA fast/slow lengths / offset (hundredths) / sigma for ALMA crossover strategy
alma_fast_lengths = list(range(2, 40, 1))
alma_slow_lengths = list(range(60, 100, 1))
alma_offsets = list(range(10, 20, 2))
alma_sigmas = list(range(1, 10, 2))


def validate_triple_ema_params(ema1_period, ema2_period, ema3_period):
    ema1_period = int(ema1_period)
    ema2_period = int(ema2_period)
    ema3_period = int(ema3_period)
    if ema1_period <= 0 or ema2_period <= 0 or ema3_period <= 0:
        raise ValueError("EMA periods must be positive integers.")
    if not (ema1_period <= ema2_period <= ema3_period):
        raise ValueError("EMA periods must be ordered as EMA1 <= EMA2 <= EMA3.")
    return ema1_period, ema2_period, ema3_period


_ema_series_cache = {}


def get_ema_series(close_series, period):
    cache_key = (id(close_series), int(period))
    if cache_key not in _ema_series_cache:
        ema = vbt.MA.run(close_series, int(period), ewm=True).ma
        _ema_series_cache[cache_key] = pd.Series(ema.values.flatten(), index=close_series.index)
    return _ema_series_cache[cache_key]


def build_triple_ema_signals(close_series, params, shift_signals=True):
    ema1_period, ema2_period, ema3_period = validate_triple_ema_params(
        params["ema1_period"],
        params["ema2_period"],
        params["ema3_period"]
    )
    ema1 = get_ema_series(close_series, ema1_period)
    ema2 = get_ema_series(close_series, ema2_period)
    ema3 = get_ema_series(close_series, ema3_period)

    entries = (
        ema1.vbt.crossed_above(ema2) |
        ema1.vbt.crossed_above(ema3) |
        ema2.vbt.crossed_above(ema3)
    ).reindex(close_series.index).fillna(False)
    exits = (
        ema1.vbt.crossed_below(ema2) |
        ema1.vbt.crossed_below(ema3) |
        ema2.vbt.crossed_below(ema3)
    ).reindex(close_series.index).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool)
    )


def format_triple_ema_params(params):
    return f"EMA({params['ema1_period']},{params['ema2_period']},{params['ema3_period']})"


def validate_macd_params(fast_period, slow_period, signal_period):
    fast_period = int(fast_period)
    slow_period = int(slow_period)
    signal_period = int(signal_period)
    if fast_period <= 0 or slow_period <= 0 or signal_period <= 0:
        raise ValueError("MACD periods must be positive integers.")
    if fast_period >= slow_period:
        raise ValueError("MACD fast period must be less than slow period.")
    return fast_period, slow_period, signal_period


def compute_macd(close_series, fast_period, slow_period, signal_period):
    fast_period, slow_period, signal_period = validate_macd_params(
        fast_period,
        slow_period,
        signal_period
    )
    close_series = pd.Series(close_series, index=close_series.index).astype(float)
    if close_series.dropna().shape[0] < slow_period + signal_period:
        raise ValueError("Not enough price bars to compute MACD for the selected periods.")

    macd_line_arr, macd_signal_arr, macd_hist_arr = talib.MACD(
        close_series.to_numpy(dtype=float),
        fastperiod=fast_period,
        slowperiod=slow_period,
        signalperiod=signal_period
    )
    return (
        pd.Series(macd_line_arr, index=close_series.index),
        pd.Series(macd_signal_arr, index=close_series.index),
        pd.Series(macd_hist_arr, index=close_series.index)
    )


def build_macd_signals(close_series, params, shift_signals=True):
    macd_line, macd_signal_line, macd_hist = compute_macd(
        close_series,
        params["macd_fast_period"],
        params["macd_slow_period"],
        params["macd_signal_period"]
    )
    entries = macd_line.vbt.crossed_above(macd_signal_line).reindex(close_series.index).fillna(False)
    exits = macd_line.vbt.crossed_below(macd_signal_line).reindex(close_series.index).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool)
    )


def format_macd_params(params):
    return (
        f"MACD({params['macd_fast_period']},"
        f"{params['macd_slow_period']},"
        f"{params['macd_signal_period']})"
    )


# AROON Up/Down crossover strategy (TA-Lib AROON timeperiod)
def validate_aroon_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"AROON timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("AROON timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "AROON timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("AROON timeperiod must be a positive integer.")
    return timeperiod


def validate_aroon_entry_level(level):
    try:
        level_num = float(level)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"AROON entry level must be numeric; got level={level!r}") from exc
    if not np.isfinite(level_num):
        raise ValueError("AROON entry level must be a finite number.")
    if not 0.0 <= level_num <= 100.0:
        raise ValueError(
            f"AROON entry level must be between 0 and 100 (Aroon scale); got {level_num}."
        )
    return level_num


AROON_ENTRY_LEVEL = validate_aroon_entry_level(AROON_ENTRY_LEVEL)


_aroon_ohlc_cache = {}
_aroon_series_cache = {}


def get_aroon_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError("stock_data/TICKER are missing. Run cell 2 (download) first.")
    if "_select_ohlc_column" not in globals():
        raise ValueError(
            "Shared OHLC helper `_select_ohlc_column` is missing. "
            "Run cell 5 (Define Indicator Grids) fully (SUPERTREND section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("AROON close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _aroon_ohlc_cache:
        return _aroon_ohlc_cache[cache_key]

    price_index = pd.DatetimeIndex(close_series.index)
    if price_index.tz is not None:
        price_index = price_index.tz_localize(None)

    high = _select_ohlc_column(stock_data, TICKER, "High")
    low = _select_ohlc_column(stock_data, TICKER, "Low")
    if isinstance(high.index, pd.DatetimeIndex) and high.index.tz is not None:
        high = high.copy()
        high.index = high.index.tz_localize(None)
    if isinstance(low.index, pd.DatetimeIndex) and low.index.tz is not None:
        low = low.copy()
        low.index = low.index.tz_localize(None)

    high = high.reindex(price_index)
    low = low.reindex(price_index)
    close = pd.Series(
        pd.Series(close_series).astype(float).to_numpy(),
        index=price_index,
        name=getattr(close_series, "name", None),
    )

    finite_close = np.isfinite(close.to_numpy(dtype=float))
    if not finite_close.any():
        raise ValueError("AROON close_series contains no finite prices.")
    valid_mask = pd.Series(finite_close, index=close.index)
    high_vals = high.loc[valid_mask].to_numpy(dtype=float)
    low_vals = low.loc[valid_mask].to_numpy(dtype=float)
    if (not np.isfinite(high_vals).all()) or (not np.isfinite(low_vals).all()):
        raise ValueError(
            "High/Low data is missing or non-finite for one or more dates in the selected price series."
        )
    if (high_vals < low_vals).any():
        raise ValueError("AROON requires High >= Low on all valid close dates.")

    _aroon_ohlc_cache[cache_key] = (high, low, close)
    return _aroon_ohlc_cache[cache_key]


def compute_aroon(close_series, timeperiod):
    timeperiod = validate_aroon_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("AROON close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _aroon_series_cache:
        return _aroon_series_cache[cache_key]

    high, low, close = get_aroon_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= timeperiod:
        raise ValueError("Not enough price bars to compute AROON for the selected timeperiod.")

    try:
        aroon_down_arr, aroon_up_arr = talib.AROON(
            high.to_numpy(dtype=float),
            low.to_numpy(dtype=float),
            timeperiod=timeperiod,
        )
    except Exception as exc:
        raise ValueError(
            f"talib.AROON failed for timeperiod={timeperiod}: {type(exc).__name__}: {exc}"
        ) from exc

    aroon_up = pd.Series(aroon_up_arr, index=close.index)
    aroon_down = pd.Series(aroon_down_arr, index=close.index)
    if (~(aroon_up.isna() | aroon_down.isna())).sum() == 0:
        raise ValueError("AROON could not be computed for the selected parameter combination.")

    _aroon_series_cache[cache_key] = (aroon_up, aroon_down)
    return _aroon_series_cache[cache_key]


def build_aroon_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for AROON, got {type(params).__name__}.")
    missing_keys = [key for key in ("aroon_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"AROON params missing keys: {missing_keys}")
    if "AROON_ENTRY_LEVEL" not in globals():
        raise ValueError("AROON_ENTRY_LEVEL is missing. Re-run cell 5 (Define Indicator Grids).")
    entry_level = validate_aroon_entry_level(AROON_ENTRY_LEVEL)

    aroon_up, aroon_down = compute_aroon(close_series, params["aroon_timeperiod"])
    # Require both lines valid on this bar and the prior bar so TA-Lib warmup NaNs
    # do not register as false Up/Down crosses (same pattern as KAMA/RSI).
    valid = aroon_up.notna() & aroon_down.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Entry trigger: Up crosses above Down, and Up is strictly above the entry level.
    entries = (
        aroon_up.vbt.crossed_above(aroon_down)
        & (aroon_up > entry_level)
        & valid_cross
    ).fillna(False)
    exits = (aroon_down.vbt.crossed_above(aroon_up) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_aroon_params(params):
    return f"AROON({params['aroon_timeperiod']})"


def validate_aroon_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_aroon_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "AROON sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("AROON sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# STC Schaff Trend Cycle strategy (EMA short/long + cycle period)
def validate_stc_params(ema_short, ema_long, cycle_period):
    ema_short = int(ema_short)
    ema_long = int(ema_long)
    cycle_period = int(cycle_period)
    if ema_short <= 0 or ema_long <= 0 or cycle_period <= 0:
        raise ValueError("STC periods must be positive integers.")
    if ema_short >= ema_long:
        raise ValueError("STC short EMA period must be less than long EMA period.")
    return ema_short, ema_long, cycle_period


def compute_stc(close_series, ema_short, ema_long, cycle_period):
    ema_short, ema_long, cycle_period = validate_stc_params(ema_short, ema_long, cycle_period)
    close = np.asarray(close_series, dtype=float)
    stc_ema_short = talib.EMA(close, timeperiod=ema_short)
    stc_ema_long = talib.EMA(close, timeperiod=ema_long)
    stc_macd = stc_ema_short - stc_ema_long

    stc_macd_highest = talib.MAX(stc_macd, timeperiod=cycle_period)
    stc_macd_lowest = talib.MIN(stc_macd, timeperiod=cycle_period)
    stc_macd_denom = stc_macd_highest - stc_macd_lowest
    with np.errstate(divide="ignore", invalid="ignore"):
        stc_k_macd = np.where(
            np.abs(stc_macd_denom) > 0,
            100.0 * (stc_macd - stc_macd_lowest) / stc_macd_denom,
            0.0,
        )
    stc_d_macd = talib.EMA(stc_k_macd, timeperiod=cycle_period)

    stc_d_highest = talib.MAX(stc_d_macd, timeperiod=cycle_period)
    stc_d_lowest = talib.MIN(stc_d_macd, timeperiod=cycle_period)
    stc_d_denom = stc_d_highest - stc_d_lowest
    with np.errstate(divide="ignore", invalid="ignore"):
        stc = np.where(
            np.abs(stc_d_denom) > 0,
            100.0 * (stc_d_macd - stc_d_lowest) / stc_d_denom,
            0.0,
        )
    stc = pd.Series(stc, index=close_series.index).replace([np.inf, -np.inf], np.nan)
    return stc


def build_stc_signals_from_series(
    stc,
    buy_level=STC_BUY_LEVEL,
    sell_level=STC_SELL_LEVEL,
    shift_signals=True,
):
    entries = (stc > buy_level) & (stc.shift(1) <= buy_level)
    exits = (stc < sell_level) & (stc.shift(1) >= sell_level)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=stc.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=stc.index, dtype=bool),
    )


def build_stc_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for STC, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("stc_ema_short_period", "stc_ema_long_period", "stc_cycle_period")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"STC params missing keys: {missing_keys}")

    ema_short, ema_long, cycle_period = validate_stc_params(
        params["stc_ema_short_period"],
        params["stc_ema_long_period"],
        params["stc_cycle_period"],
    )
    stc = compute_stc(close_series, ema_short, ema_long, cycle_period)
    if stc.notna().sum() == 0:
        raise ValueError("STC could not be computed for the selected parameter combination.")
    return build_stc_signals_from_series(stc, shift_signals=shift_signals)


def format_stc_params(params):
    return (
        f"STC({params['stc_ema_short_period']},"
        f"{params['stc_ema_long_period']},"
        f"{params['stc_cycle_period']})"
    )


# KAMA price crossover strategy (custom KAMA: period + fastest/slowest EMA constants)
if (
    "validate_kama_params" not in globals()
    or "_compute_kama_core" not in globals()
):
    raise ValueError("KAMA helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_kama_series_cache = {}


def get_kama_series(close_series, period, fast_ema_constant, slow_ema_constant):
    period, fast_ema_constant, slow_ema_constant = validate_kama_params(
        period,
        fast_ema_constant,
        slow_ema_constant,
    )
    cache_key = (id(close_series), period, fast_ema_constant, slow_ema_constant)
    if cache_key not in _kama_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        kama_line = _compute_kama_core(
            close,
            period,
            fast_ema_constant,
            slow_ema_constant,
        )
        if kama_line.notna().sum() == 0:
            raise ValueError("KAMA could not be computed for the selected parameter combination.")
        _kama_series_cache[cache_key] = (close, kama_line)
    return _kama_series_cache[cache_key]


def build_kama_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for KAMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("kama_period", "kama_fast_ema_constant", "kama_slow_ema_constant")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"KAMA params missing keys: {missing_keys}")

    close, kama_line = get_kama_series(
        close_series,
        params["kama_period"],
        params["kama_fast_ema_constant"],
        params["kama_slow_ema_constant"],
    )
    valid_kama = kama_line.notna()
    valid_cross = valid_kama & valid_kama.shift(1).fillna(False)
    entries = (close.vbt.crossed_above(kama_line) & valid_cross).fillna(False)
    exits = (close.vbt.crossed_below(kama_line) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_kama_params(params):
    return (
        f"KAMA({params['kama_period']},"
        f"{params['kama_fast_ema_constant']},"
        f"{params['kama_slow_ema_constant']})"
    )


# SUPERTREND trend-flip strategy (ATR length + factor/multiplier via TA-Lib ATR)
if (
    "validate_supertrend_params" not in globals()
    or "_compute_supertrend_core" not in globals()
):
    raise ValueError("SUPERTREND helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_supertrend_ohlc_cache = {}
_supertrend_series_cache = {}


def _select_ohlc_column(df, ticker, column_name):
    if isinstance(df.columns, pd.MultiIndex):
        key = (column_name, ticker)
        if key in df.columns:
            series = df[key]
        else:
            # Exact level match only — avoid substring false positives like "HigherHigh"
            cols = [
                c for c in df.columns
                if isinstance(c, tuple) and len(c) >= 1 and c[0] == column_name
            ]
            if not cols:
                raise KeyError(f"{column_name} not found for ticker {ticker!r}")
            series = df[cols[0]]
    else:
        if column_name not in df.columns:
            raise KeyError(f"{column_name} not found in stock_data columns")
        series = df[column_name]
    return pd.Series(series.astype(float).to_numpy(), index=series.index, name=column_name)


def get_supertrend_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError("stock_data/TICKER are missing. Run cell 2 (download) first.")
    if close_series is None or len(close_series) == 0:
        raise ValueError("SUPERTREND close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _supertrend_ohlc_cache:
        return _supertrend_ohlc_cache[cache_key]

    high = _select_ohlc_column(stock_data, TICKER, "High").reindex(close_series.index)
    low = _select_ohlc_column(stock_data, TICKER, "Low").reindex(close_series.index)
    close = pd.Series(close_series, index=close_series.index).astype(float)

    valid_close = close.notna()
    if not valid_close.any():
        raise ValueError("SUPERTREND close_series contains no finite prices.")
    if high.loc[valid_close].isna().any() or low.loc[valid_close].isna().any():
        raise ValueError(
            "High/Low data is missing for one or more dates in the selected price series."
        )
    if (high.loc[valid_close] < low.loc[valid_close]).any():
        raise ValueError("SUPERTREND requires High >= Low on all valid close dates.")

    _supertrend_ohlc_cache[cache_key] = (high, low, close)
    return _supertrend_ohlc_cache[cache_key]


def get_supertrend_series(close_series, atr_length, factor):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    cache_key = (id(close_series), atr_length, factor)
    if cache_key not in _supertrend_series_cache:
        high, low, close = get_supertrend_ohlc(close_series)
        supertrend_line, direction = _compute_supertrend_core(
            high.to_numpy(dtype=float),
            low.to_numpy(dtype=float),
            close.to_numpy(dtype=float),
            atr_length,
            factor,
        )
        supertrend_line = pd.Series(supertrend_line, index=close.index)
        direction = pd.Series(direction, index=close.index)
        if direction.notna().sum() == 0:
            raise ValueError("SUPERTREND could not be computed for the selected parameter combination.")
        _supertrend_series_cache[cache_key] = (close, supertrend_line, direction)
    return _supertrend_series_cache[cache_key]


def build_supertrend_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for SUPERTREND, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("supertrend_atr_length", "supertrend_factor")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"SUPERTREND params missing keys: {missing_keys}")

    close, _supertrend_line, direction = get_supertrend_series(
        close_series,
        params["supertrend_atr_length"],
        params["supertrend_factor"],
    )
    valid_dir = direction.notna()
    valid_flip = valid_dir & valid_dir.shift(1).fillna(False)
    # Buy: close above red line → direction flips to green (+1)
    entries = ((direction == 1.0) & (direction.shift(1) == -1.0) & valid_flip).fillna(False)
    # Sell: close below green line → direction flips to red (-1)
    exits = ((direction == -1.0) & (direction.shift(1) == 1.0) & valid_flip).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_supertrend_params(params):
    return (
        f"SUPERTREND(ATR={params['supertrend_atr_length']},"
        f"Factor={params['supertrend_factor']})"
    )


def validate_supertrend_sensitivity_params(atr_length, factor, factor_anchor):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    if int(factor_anchor) != factor:
        raise ValueError("SUPERTREND sensitivity anchor must match the factor.")
    return atr_length, factor, factor


# KALMAN price vs filter strategy (Q=process noise, R=measurement noise)
if (
    "validate_kalman_params" not in globals()
    or "_compute_kalman_core" not in globals()
):
    raise ValueError("KALMAN helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_kalman_series_cache = {}


def get_kalman_series(close_series, process_noise, measurement_noise):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if close_series is None or len(close_series) == 0:
        raise ValueError("KALMAN close_series is empty.")
    cache_key = (id(close_series), process_noise, measurement_noise)
    if cache_key not in _kalman_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        close_values = close.to_numpy(dtype=float)
        if not np.isfinite(close_values).any():
            raise ValueError("KALMAN close_series contains no finite prices.")
        kalman_line = pd.Series(
            _compute_kalman_core(close_values, process_noise, measurement_noise),
            index=close.index,
        )
        if kalman_line.notna().sum() == 0:
            raise ValueError("KALMAN could not be computed for the selected parameter combination.")
        _kalman_series_cache[cache_key] = (close, kalman_line)
    return _kalman_series_cache[cache_key]


def build_kalman_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for KALMAN, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("kalman_process_noise", "kalman_measurement_noise")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"KALMAN params missing keys: {missing_keys}")

    close, kalman_line = get_kalman_series(
        close_series,
        params["kalman_process_noise"],
        params["kalman_measurement_noise"],
    )
    valid = kalman_line.notna() & close.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: price closes above the Kalman filter line (cross into above)
    entries = (close.vbt.crossed_above(kalman_line) & valid_cross).fillna(False)
    # Sell: price closes below the Kalman filter line (cross into below)
    exits = (close.vbt.crossed_below(kalman_line) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_kalman_params(params):
    return (
        f"KALMAN(Q={params['kalman_process_noise']},"
        f"R={params['kalman_measurement_noise']})"
    )


def validate_kalman_sensitivity_params(process_noise, measurement_noise, measurement_noise_anchor):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if int(measurement_noise_anchor) != measurement_noise:
        raise ValueError("KALMAN sensitivity anchor must match the measurement noise (R).")
    return process_noise, measurement_noise, measurement_noise


# RSI midline crossover strategy (TA-Lib RSI timeperiod; buy/sell on cross of 50)
def validate_rsi_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"RSI timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("RSI timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "RSI timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("RSI timeperiod must be a positive integer.")
    return timeperiod


_rsi_series_cache = {}


def compute_rsi(close_series, timeperiod):
    timeperiod = validate_rsi_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("RSI close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _rsi_series_cache:
        return _rsi_series_cache[cache_key]

    close = pd.Series(close_series, index=close_series.index).astype(float)
    close_values = close.to_numpy(dtype=float)
    if not np.isfinite(close_values).any():
        raise ValueError("RSI close_series contains no finite prices.")
    if np.isfinite(close_values).sum() < timeperiod + 1:
        raise ValueError("Not enough price bars to compute RSI for the selected timeperiod.")

    rsi = pd.Series(
        talib.RSI(close_values, timeperiod=timeperiod),
        index=close.index,
    )
    if rsi.notna().sum() == 0:
        raise ValueError("RSI could not be computed for the selected parameter combination.")
    _rsi_series_cache[cache_key] = rsi
    return rsi


def build_rsi_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for RSI, got {type(params).__name__}.")
    missing_keys = [key for key in ("rsi_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"RSI params missing keys: {missing_keys}")

    rsi = compute_rsi(close_series, params["rsi_timeperiod"])
    valid = rsi.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: RSI crosses above 50 (rising bullish momentum)
    entries = (
        (rsi > RSI_CROSS_LEVEL) & (rsi.shift(1) <= RSI_CROSS_LEVEL) & valid_cross
    ).fillna(False)
    # Sell: RSI crosses below 50 (increasing bearish momentum)
    exits = (
        (rsi < RSI_CROSS_LEVEL) & (rsi.shift(1) >= RSI_CROSS_LEVEL) & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=rsi.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=rsi.index, dtype=bool),
    )


def format_rsi_params(params):
    return f"RSI({params['rsi_timeperiod']})"


def validate_rsi_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_rsi_params(timeperiod)
    if int(timeperiod_anchor_a) != timeperiod or int(timeperiod_anchor_b) != timeperiod:
        raise ValueError("RSI sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod



# ADX +DI/-DI crossover strategy (TA-Lib PLUS_DI / MINUS_DI; ADX = Wilder-smoothed DX)
def validate_adx_params(di_length, adx_smoothing):
    try:
        di_length_num = float(di_length)
        adx_smoothing_num = float(adx_smoothing)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"ADX params must be numeric; got di_length={di_length!r}, "
            f"adx_smoothing={adx_smoothing!r}"
        ) from exc
    if not np.isfinite(di_length_num) or not np.isfinite(adx_smoothing_num):
        raise ValueError("ADX DI length and ADX smoothing must be finite numbers.")
    if di_length_num != int(di_length_num):
        raise ValueError(
            "ADX DI length must be a whole number for this grid "
            f"(sensitivity uses int grids); got di_length={di_length!r}"
        )
    if adx_smoothing_num != int(adx_smoothing_num):
        raise ValueError(
            "ADX smoothing timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got adx_smoothing={adx_smoothing!r}"
        )
    di_length = int(di_length_num)
    adx_smoothing = int(adx_smoothing_num)
    if di_length <= 0 or adx_smoothing <= 0:
        raise ValueError("ADX DI length and ADX smoothing must be positive integers.")
    return di_length, adx_smoothing


def validate_adx_trend_level(level):
    try:
        level_num = float(level)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"ADX trend level must be numeric; got level={level!r}") from exc
    if not np.isfinite(level_num):
        raise ValueError("ADX trend level must be a finite number.")
    if not 0.0 <= level_num <= 100.0:
        raise ValueError(
            f"ADX trend level must be between 0 and 100 (ADX scale); got {level_num}."
        )
    return level_num


ADX_TREND_LEVEL = validate_adx_trend_level(ADX_TREND_LEVEL)


_adx_ohlc_cache = {}
_adx_di_cache = {}
_adx_series_cache = {}


def get_adx_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError("stock_data/TICKER are missing. Run cell 2 (download) first.")
    if "_select_ohlc_column" not in globals():
        raise ValueError(
            "Shared OHLC helper `_select_ohlc_column` is missing. "
            "Run cell 5 (Define Indicator Grids) fully (SUPERTREND section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _adx_ohlc_cache:
        return _adx_ohlc_cache[cache_key]

    price_index = pd.DatetimeIndex(close_series.index)
    if price_index.tz is not None:
        price_index = price_index.tz_localize(None)

    high = _select_ohlc_column(stock_data, TICKER, "High")
    low = _select_ohlc_column(stock_data, TICKER, "Low")
    if isinstance(high.index, pd.DatetimeIndex) and high.index.tz is not None:
        high = high.copy()
        high.index = high.index.tz_localize(None)
    if isinstance(low.index, pd.DatetimeIndex) and low.index.tz is not None:
        low = low.copy()
        low.index = low.index.tz_localize(None)

    high = high.reindex(price_index)
    low = low.reindex(price_index)
    close = pd.Series(
        pd.Series(close_series).astype(float).to_numpy(),
        index=price_index,
        name=getattr(close_series, "name", None),
    )

    finite_close = np.isfinite(close.to_numpy(dtype=float))
    if not finite_close.any():
        raise ValueError("ADX close_series contains no finite prices.")
    valid_mask = pd.Series(finite_close, index=close.index)
    high_vals = high.loc[valid_mask].to_numpy(dtype=float)
    low_vals = low.loc[valid_mask].to_numpy(dtype=float)
    if (not np.isfinite(high_vals).all()) or (not np.isfinite(low_vals).all()):
        raise ValueError(
            "High/Low data is missing or non-finite for one or more dates in the selected price series."
        )
    if (high_vals < low_vals).any():
        raise ValueError("ADX requires High >= Low on all valid close dates.")

    _adx_ohlc_cache[cache_key] = (high, low, close)
    return _adx_ohlc_cache[cache_key]


def _compute_adx_di_lines(close_series, di_length):
    """TA-Lib +DI/-DI for a DI length; cached so each length is computed once per series."""
    di_length = validate_adx_params(di_length, di_length)[0]
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")
    cache_key = (id(close_series), di_length)
    if cache_key in _adx_di_cache:
        return _adx_di_cache[cache_key]

    high, low, close = get_adx_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= di_length:
        raise ValueError("Not enough price bars to compute ADX DI lines for the selected DI length.")

    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    close_vals = close.to_numpy(dtype=float)
    try:
        plus_di_arr = talib.PLUS_DI(high_vals, low_vals, close_vals, timeperiod=di_length)
        minus_di_arr = talib.MINUS_DI(high_vals, low_vals, close_vals, timeperiod=di_length)
    except Exception as exc:
        raise ValueError(
            f"talib PLUS_DI/MINUS_DI failed for di_length={di_length}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    plus_di = pd.Series(plus_di_arr, index=close.index)
    minus_di = pd.Series(minus_di_arr, index=close.index)
    if (~(plus_di.isna() | minus_di.isna())).sum() == 0:
        raise ValueError("ADX DI lines could not be computed for the selected DI length.")

    _adx_di_cache[cache_key] = (plus_di, minus_di, close)
    return _adx_di_cache[cache_key]


def _wilder_smooth_dx(dx_values, adx_smoothing):
    """Wilder/RMA smooth of DX -> ADX (same DI lines for any DI length / smoothing pair)."""
    dx = np.asarray(dx_values, dtype=float)
    adx = np.full(dx.shape, np.nan, dtype=float)
    if adx_smoothing <= 0:
        raise ValueError("ADX smoothing must be a positive integer.")
    n = int(dx.size)
    if n == 0:
        return adx

    # Seed with SMA of the first contiguous `adx_smoothing` finite DX values,
    # then Wilder-smooth — common ADX / TA-Lib-style initialization.
    run = 0
    seed_end = None
    for i in range(n):
        if np.isfinite(dx[i]):
            run += 1
            if run == adx_smoothing:
                seed_end = i
                break
        else:
            run = 0
    if seed_end is None:
        return adx

    seed_start = seed_end - adx_smoothing + 1
    prev = float(np.mean(dx[seed_start : seed_end + 1]))
    adx[seed_end] = prev
    for i in range(seed_end + 1, n):
        if not np.isfinite(dx[i]):
            adx[i] = np.nan
            continue
        prev = ((prev * (adx_smoothing - 1)) + float(dx[i])) / adx_smoothing
        adx[i] = prev
    return adx


def compute_adx(close_series, di_length, adx_smoothing):
    di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")
    cache_key = (id(close_series), di_length, adx_smoothing)
    if cache_key in _adx_series_cache:
        return _adx_series_cache[cache_key]

    plus_di, minus_di, close = _compute_adx_di_lines(close_series, di_length)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    # DI warmup + ADX seed window
    min_bars = di_length + adx_smoothing
    if finite_bars < min_bars:
        raise ValueError(
            "Not enough price bars to compute ADX for the selected parameter combination "
            f"(need at least di_length + adx_smoothing = {min_bars} finite bars)."
        )

    plus_vals = plus_di.to_numpy(dtype=float)
    minus_vals = minus_di.to_numpy(dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        denom = plus_vals + minus_vals
        dx_vals = np.where(
            np.isfinite(denom) & (denom != 0.0),
            100.0 * np.abs(plus_vals - minus_vals) / denom,
            np.nan,
        )
    if not np.isfinite(dx_vals).any():
        raise ValueError("ADX DX could not be computed for the selected parameter combination.")

    try:
        adx_arr = _wilder_smooth_dx(dx_vals, adx_smoothing)
    except Exception as exc:
        raise ValueError(
            f"ADX smoothing failed for di_length={di_length}, "
            f"adx_smoothing={adx_smoothing}: {type(exc).__name__}: {exc}"
        ) from exc

    adx = pd.Series(adx_arr, index=close.index)
    if (~(plus_di.isna() | minus_di.isna() | adx.isna())).sum() == 0:
        raise ValueError("ADX could not be computed for the selected parameter combination.")

    _adx_series_cache[cache_key] = (plus_di, minus_di, adx)
    return _adx_series_cache[cache_key]


def build_adx_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ADX, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("adx_di_length", "adx_smoothing_timeperiod")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ADX params missing keys: {missing_keys}")
    if "ADX_TREND_LEVEL" not in globals():
        raise ValueError("ADX_TREND_LEVEL is missing. Re-run cell 5 (Define Indicator Grids).")
    trend_level = validate_adx_trend_level(ADX_TREND_LEVEL)

    plus_di, minus_di, adx = compute_adx(
        close_series,
        params["adx_di_length"],
        params["adx_smoothing_timeperiod"],
    )
    if len(plus_di) != len(close_series):
        raise ValueError(
            "ADX output length does not match close_series; check OHLC alignment."
        )
    # Require DI lines and ADX valid on this bar and the prior bar so warmup NaNs
    # do not register as false +DI/-DI crosses (same pattern as AROON/KAMA/RSI).
    valid = plus_di.notna() & minus_di.notna() & adx.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: +DI crosses above -DI while ADX confirms a strong uptrend
    entries = (
        plus_di.vbt.crossed_above(minus_di)
        & (adx > trend_level)
        & valid_cross
    ).fillna(False)
    # Sell: -DI crosses above +DI while ADX confirms a strong downtrend
    exits = (
        minus_di.vbt.crossed_above(plus_di)
        & (adx > trend_level)
        & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_adx_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ADX, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("adx_di_length", "adx_smoothing_timeperiod")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ADX params missing keys: {missing_keys}")
    return (
        f"ADX(DI={params['adx_di_length']},"
        f"Smooth={params['adx_smoothing_timeperiod']})"
    )


def validate_adx_sensitivity_params(di_length, adx_smoothing, adx_smoothing_anchor):
    di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
    try:
        anchor = int(adx_smoothing_anchor)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "ADX sensitivity anchor must be an integer matching the ADX smoothing timeperiod."
        ) from exc
    if anchor != adx_smoothing:
        raise ValueError("ADX sensitivity anchor must match the ADX smoothing timeperiod.")
    return di_length, adx_smoothing, adx_smoothing


# DONCHIAN upper-band breakout strategy (TA-Lib MAX/MIN timeperiod; sell on middle-band cross)
def validate_donchian_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"DONCHIAN timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("DONCHIAN timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "DONCHIAN timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("DONCHIAN timeperiod must be a positive integer.")
    return timeperiod


_donchian_series_cache = {}


def get_donchian_ohlc(close_series):
    """Reuse AROON's High/Low alignment cache — same extract, no second OHLC pass."""
    if "get_aroon_ohlc" not in globals():
        raise ValueError(
            "Shared OHLC helper get_aroon_ohlc is missing. "
            "Run cell 5 (Define Indicator Grids) fully (AROON section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("DONCHIAN close_series is empty.")
    try:
        return get_aroon_ohlc(close_series)
    except ValueError as exc:
        msg = str(exc)
        if msg.startswith("AROON "):
            raise ValueError("DONCHIAN " + msg[len("AROON "):]) from exc
        if "AROON requires" in msg:
            raise ValueError(msg.replace("AROON requires", "DONCHIAN requires", 1)) from exc
        raise


def compute_donchian(close_series, timeperiod):
    timeperiod = validate_donchian_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("DONCHIAN close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _donchian_series_cache:
        return _donchian_series_cache[cache_key]

    high, low, close = get_donchian_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= timeperiod:
        raise ValueError("Not enough price bars to compute DONCHIAN for the selected timeperiod.")

    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    try:
        # TA-Lib has no DONCHIAN; upper/lower from MAX/MIN over High/Low.
        upper_arr = talib.MAX(high_vals, timeperiod=timeperiod)
        lower_arr = talib.MIN(low_vals, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib MAX/MIN failed for DONCHIAN timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    upper = pd.Series(upper_arr, index=close.index)
    lower = pd.Series(lower_arr, index=close.index)
    middle = (upper + lower) / 2.0
    if (~(upper.isna() | lower.isna() | middle.isna())).sum() == 0:
        raise ValueError("DONCHIAN could not be computed for the selected parameter combination.")

    _donchian_series_cache[cache_key] = (close, upper, middle, lower)
    return _donchian_series_cache[cache_key]


def build_donchian_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for DONCHIAN, got {type(params).__name__}.")
    missing_keys = [key for key in ("donchian_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"DONCHIAN params missing keys: {missing_keys}")

    close, upper, middle, _lower = compute_donchian(
        close_series, params["donchian_timeperiod"]
    )
    if len(close) != len(close_series):
        raise ValueError(
            "DONCHIAN output length does not match close_series; check OHLC alignment."
        )
    # Use prior bar's channel so close can break above upper (same-bar MAX(High)
    # embeds today's high and makes close > upper impossible).
    prior_upper = upper.shift(1)
    prior_middle = middle.shift(1)
    valid = prior_upper.notna() & prior_middle.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: close crosses above the prior upper band (uptrend breakout)
    entries = (close.vbt.crossed_above(prior_upper) & valid_cross).fillna(False)
    # Sell: close crosses below the prior middle band (trailing stop)
    exits = (close.vbt.crossed_below(prior_middle) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX pattern). close.index may be
    # tz-stripped inside get_aroon_ohlc and would mis-align the backtest.
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_donchian_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for DONCHIAN, got {type(params).__name__}.")
    missing_keys = [key for key in ("donchian_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"DONCHIAN params missing keys: {missing_keys}")
    return f"DONCHIAN({params['donchian_timeperiod']})"


def validate_donchian_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_donchian_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "DONCHIAN sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("DONCHIAN sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# TRIX zero-line crossover strategy (TA-Lib TRIX timeperiod; buy/sell on cross of 0)
def validate_trix_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"TRIX timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("TRIX timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "TRIX timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("TRIX timeperiod must be a positive integer.")
    return timeperiod


_trix_series_cache = {}


def compute_trix(close_series, timeperiod):
    timeperiod = validate_trix_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("TRIX close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _trix_series_cache:
        return _trix_series_cache[cache_key]

    close = pd.Series(close_series, index=close_series.index).astype(float)
    close_values = close.to_numpy(dtype=float)
    if not np.isfinite(close_values).any():
        raise ValueError("TRIX close_series contains no finite prices.")

    # TRIX is ROC of a triple EMA; needs far more warmup than timeperiod+1.
    # TA-Lib TRIX lookback is 3*(timeperiod-1)+1; need one more bar for a usable value.
    min_bars = 3 * (timeperiod - 1) + 2
    if np.isfinite(close_values).sum() < min_bars:
        raise ValueError(
            "Not enough price bars to compute TRIX for the selected timeperiod "
            f"(need >= {min_bars} finite bars for timeperiod={timeperiod})."
        )

    try:
        trix_arr = talib.TRIX(close_values, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib.TRIX failed for timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    trix = pd.Series(trix_arr, index=close.index)
    if trix.notna().sum() == 0:
        raise ValueError("TRIX could not be computed for the selected parameter combination.")
    _trix_series_cache[cache_key] = trix
    return trix


def build_trix_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for TRIX, got {type(params).__name__}.")
    missing_keys = [key for key in ("trix_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"TRIX params missing keys: {missing_keys}")
    if "TRIX_CROSS_LEVEL" not in globals():
        raise ValueError("TRIX_CROSS_LEVEL is missing. Re-run cell 5 (Define Indicator Grids).")

    trix = compute_trix(close_series, params["trix_timeperiod"])
    valid = trix.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: TRIX crosses above 0 (bullish momentum)
    entries = (
        (trix > TRIX_CROSS_LEVEL) & (trix.shift(1) <= TRIX_CROSS_LEVEL) & valid_cross
    ).fillna(False)
    # Sell: TRIX crosses below 0 (bearish momentum)
    exits = (
        (trix < TRIX_CROSS_LEVEL) & (trix.shift(1) >= TRIX_CROSS_LEVEL) & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_trix_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for TRIX, got {type(params).__name__}.")
    missing_keys = [key for key in ("trix_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"TRIX params missing keys: {missing_keys}")
    return f"TRIX({params['trix_timeperiod']})"


def validate_trix_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_trix_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "TRIX sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("TRIX sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# VORTEX VI+/VI- crossover strategy (TA-Lib has no VORTEX; TRANGE/SUM over timeperiod)
def validate_vortex_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"VORTEX timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("VORTEX timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "VORTEX timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("VORTEX timeperiod must be a positive integer.")
    return timeperiod


_vortex_series_cache = {}
_vortex_base_cache = {}


def get_vortex_ohlc(close_series):
    """Reuse AROON's High/Low alignment cache — same extract, no second OHLC pass."""
    if "get_aroon_ohlc" not in globals():
        raise ValueError(
            "Shared OHLC helper get_aroon_ohlc is missing. "
            "Run cell 5 (Define Indicator Grids) fully (AROON section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    try:
        return get_aroon_ohlc(close_series)
    except ValueError as exc:
        msg = str(exc)
        if msg.startswith("AROON "):
            raise ValueError("VORTEX " + msg[len("AROON "):]) from exc
        if "AROON requires" in msg:
            raise ValueError(msg.replace("AROON requires", "VORTEX requires", 1)) from exc
        raise


def _get_vortex_base_arrays(close_series):
    """Cache TRANGE and VM+/- once per price series (independent of timeperiod)."""
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    cache_key = id(close_series)
    if cache_key in _vortex_base_cache:
        return _vortex_base_cache[cache_key]

    high, low, close = get_vortex_ohlc(close_series)
    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    close_vals = close.to_numpy(dtype=float)
    if not (len(high_vals) == len(low_vals) == len(close_vals)):
        raise ValueError("VORTEX High/Low/Close lengths do not match after OHLC alignment.")

    try:
        # TA-Lib has no VORTEX; VI+/VI- from vortex movement vs true range.
        tr_arr = talib.TRANGE(high_vals, low_vals, close_vals)
    except Exception as exc:
        raise ValueError(
            f"talib.TRANGE failed for VORTEX: {type(exc).__name__}: {exc}"
        ) from exc

    # First bar has no prior High/Low; leave VM undefined (NaN), not 0.
    vm_plus = np.full(len(high_vals), np.nan, dtype=float)
    vm_minus = np.full(len(high_vals), np.nan, dtype=float)
    vm_plus[1:] = np.abs(high_vals[1:] - low_vals[:-1])
    vm_minus[1:] = np.abs(low_vals[1:] - high_vals[:-1])

    _vortex_base_cache[cache_key] = (tr_arr, vm_plus, vm_minus, close)
    return _vortex_base_cache[cache_key]


def compute_vortex(close_series, timeperiod):
    timeperiod = validate_vortex_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _vortex_series_cache:
        return _vortex_series_cache[cache_key]

    tr_arr, vm_plus, vm_minus, close = _get_vortex_base_arrays(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    # First VI needs a prior bar for VM+/- and `timeperiod` finite TR/VM values.
    min_bars = timeperiod + 1
    if finite_bars < min_bars:
        raise ValueError(
            "Not enough price bars to compute VORTEX for the selected timeperiod "
            f"(need >= {min_bars} finite bars for timeperiod={timeperiod})."
        )

    try:
        sum_vm_plus = talib.SUM(vm_plus, timeperiod=timeperiod)
        sum_vm_minus = talib.SUM(vm_minus, timeperiod=timeperiod)
        sum_tr = talib.SUM(tr_arr, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib.SUM failed for VORTEX timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    vi_plus_arr = np.full(len(sum_tr), np.nan, dtype=float)
    vi_minus_arr = np.full(len(sum_tr), np.nan, dtype=float)
    valid_tr = np.isfinite(sum_tr) & (sum_tr != 0)
    # Only divide where the TR sum is finite and non-zero.
    vi_plus_arr[valid_tr] = sum_vm_plus[valid_tr] / sum_tr[valid_tr]
    vi_minus_arr[valid_tr] = sum_vm_minus[valid_tr] / sum_tr[valid_tr]

    vi_plus = pd.Series(vi_plus_arr, index=close.index)
    vi_minus = pd.Series(vi_minus_arr, index=close.index)
    if (~(vi_plus.isna() | vi_minus.isna())).sum() == 0:
        raise ValueError("VORTEX could not be computed for the selected parameter combination.")

    _vortex_series_cache[cache_key] = (vi_plus, vi_minus)
    return _vortex_series_cache[cache_key]


def build_vortex_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for VORTEX, got {type(params).__name__}.")
    missing_keys = [key for key in ("vortex_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"VORTEX params missing keys: {missing_keys}")

    vi_plus, vi_minus = compute_vortex(close_series, params["vortex_timeperiod"])
    if len(vi_plus) != len(close_series) or len(vi_minus) != len(close_series):
        raise ValueError(
            "VORTEX output length does not match close_series; check OHLC alignment."
        )
    # Require both lines valid on this bar and the prior bar so warmup NaNs
    # do not register as false VI+/VI- crosses (same pattern as AROON/KAMA/RSI).
    valid = vi_plus.notna() & vi_minus.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: VI+ crosses upward from below VI-
    entries = (vi_plus.vbt.crossed_above(vi_minus) & valid_cross).fillna(False)
    # Sell: VI- crosses upward from below VI+
    exits = (vi_minus.vbt.crossed_above(vi_plus) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )

def format_vortex_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for VORTEX, got {type(params).__name__}.")
    missing_keys = [key for key in ("vortex_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"VORTEX params missing keys: {missing_keys}")
    return f"VORTEX({params['vortex_timeperiod']})"


def validate_vortex_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_vortex_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "VORTEX sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("VORTEX sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# ALMA crossover strategy (custom ALMA: fast/slow length + offset hundredths + sigma)
if (
    "validate_alma_params" not in globals()
    or "_compute_alma_core" not in globals()
):
    raise ValueError("ALMA helpers are missing. Run cell 3 (TA-Lib indicators) first.")


_alma_series_cache = {}


def validate_alma_crossover_params(fast_length, slow_length, offset, sigma):
    fast_length, offset, sigma = validate_alma_params(fast_length, offset, sigma)
    slow_length, offset, sigma = validate_alma_params(slow_length, offset, sigma)
    if fast_length >= slow_length:
        raise ValueError("ALMA fast length must be less than slow length.")
    return fast_length, slow_length, offset, sigma


def get_alma_series(close_series, length, offset, sigma):
    length, offset, sigma = validate_alma_params(length, offset, sigma)
    if close_series is None or len(close_series) == 0:
        raise ValueError("ALMA close_series is empty.")
    if not hasattr(close_series, "index"):
        raise TypeError(
            f"ALMA close_series must be a pandas Series; got {type(close_series).__name__}."
        )
    cache_key = (id(close_series), length, offset, sigma)
    if cache_key not in _alma_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        close_values = close.to_numpy(dtype=float)
        if not np.isfinite(close_values).any():
            raise ValueError("ALMA close_series contains no finite prices.")
        try:
            alma_line = pd.Series(
                _compute_alma_core(close_values, length, offset, sigma),
                index=close.index,
            )
        except ValueError as exc:
            raise ValueError(
                f"ALMA could not be computed for length={length}, offset={offset}, "
                f"sigma={sigma}: {exc}"
            ) from exc
        if alma_line.notna().sum() == 0:
            raise ValueError("ALMA could not be computed for the selected parameter combination.")
        _alma_series_cache[cache_key] = (close, alma_line)
    return _alma_series_cache[cache_key]


def build_alma_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ALMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ALMA params missing keys: {missing_keys}")

    fast_length, slow_length, offset, sigma = validate_alma_crossover_params(
        params["alma_fast_length"],
        params["alma_slow_length"],
        params["alma_offset"],
        params["alma_sigma"],
    )

    close, alma_fast = get_alma_series(close_series, fast_length, offset, sigma)
    _, alma_slow = get_alma_series(close_series, slow_length, offset, sigma)
    if len(alma_fast) != len(close_series) or len(alma_slow) != len(close_series):
        raise ValueError(
            "ALMA output length does not match close_series; check close alignment."
        )
    # Require both ALMAs and close valid on this bar and the prior bar so warmup NaNs
    # do not register as false fast/slow crosses (same pattern as AROON/KAMA/RSI).
    valid = alma_fast.notna() & alma_slow.notna() & close.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: fast ALMA crosses above slow ALMA
    entries = (alma_fast.vbt.crossed_above(alma_slow) & valid_cross).fillna(False)
    # Sell: fast ALMA crosses below slow ALMA
    exits = (alma_fast.vbt.crossed_below(alma_slow) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_alma_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ALMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ALMA params missing keys: {missing_keys}")
    fast_length, slow_length, offset, sigma = validate_alma_crossover_params(
        params["alma_fast_length"],
        params["alma_slow_length"],
        params["alma_offset"],
        params["alma_sigma"],
    )
    return f"ALMA({fast_length},{slow_length},{offset / 100:.2f},{sigma})"


ema_combinations = []
for ema1 in ema1_periods:
    for ema2 in ema2_periods:
        for ema3 in ema3_periods:
            try:
                ema1, ema2, ema3 = validate_triple_ema_params(ema1, ema2, ema3)
                ema_combinations.append({
                    "ema1_period": ema1,
                    "ema2_period": ema2,
                    "ema3_period": ema3
                })
            except ValueError:
                pass

if not ema_combinations:
    raise ValueError("No valid Triple EMA parameter combinations were generated.")

macd_combinations = []
for fast_period in macd_fast_periods:
    for slow_period in macd_slow_periods:
        for signal_period in macd_signal_periods:
            try:
                fast_period, slow_period, signal_period = validate_macd_params(
                    fast_period,
                    slow_period,
                    signal_period
                )
                macd_combinations.append({
                    "macd_fast_period": fast_period,
                    "macd_slow_period": slow_period,
                    "macd_signal_period": signal_period
                })
            except ValueError:
                pass

if not macd_combinations:
    raise ValueError("No valid MACD parameter combinations were generated.")

aroon_combinations = []
for timeperiod in aroon_timeperiods:
    try:
        timeperiod = validate_aroon_params(timeperiod)
        aroon_combinations.append({
            "aroon_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not aroon_combinations:
    raise ValueError("No valid AROON parameter combinations were generated.")

stc_combinations = []
for ema_short in stc_ema_short_periods:
    for ema_long in stc_ema_long_periods:
        for cycle_period in stc_cycle_periods:
            try:
                ema_short, ema_long, cycle_period = validate_stc_params(
                    ema_short,
                    ema_long,
                    cycle_period,
                )
                stc_combinations.append({
                    "stc_ema_short_period": ema_short,
                    "stc_ema_long_period": ema_long,
                    "stc_cycle_period": cycle_period,
                })
            except ValueError:
                pass

if not stc_combinations:
    raise ValueError("No valid STC parameter combinations were generated.")

kama_combinations = []
for period in kama_periods:
    for fast_ema_constant in kama_fast_ema_constants:
        for slow_ema_constant in kama_slow_ema_constants:
            try:
                period, fast_ema_constant, slow_ema_constant = validate_kama_params(
                    period,
                    fast_ema_constant,
                    slow_ema_constant,
                )
                kama_combinations.append({
                    "kama_period": period,
                    "kama_fast_ema_constant": fast_ema_constant,
                    "kama_slow_ema_constant": slow_ema_constant,
                })
            except ValueError:
                pass

if not kama_combinations:
    raise ValueError("No valid KAMA parameter combinations were generated.")

supertrend_combinations = []
for atr_length in supertrend_atr_lengths:
    for factor in supertrend_factors:
        try:
            atr_length, factor = validate_supertrend_params(atr_length, factor)
            supertrend_combinations.append({
                "supertrend_atr_length": atr_length,
                "supertrend_factor": factor,
            })
        except ValueError:
            pass

if not supertrend_combinations:
    raise ValueError("No valid SUPERTREND parameter combinations were generated.")

kalman_combinations = []
for process_noise in kalman_process_noise_values:
    for measurement_noise in kalman_measurement_noise_values:
        try:
            process_noise, measurement_noise = validate_kalman_params(
                process_noise,
                measurement_noise,
            )
            kalman_combinations.append({
                "kalman_process_noise": process_noise,
                "kalman_measurement_noise": measurement_noise,
            })
        except ValueError:
            pass

if not kalman_combinations:
    raise ValueError("No valid KALMAN parameter combinations were generated.")

rsi_combinations = []
for timeperiod in rsi_timeperiods:
    try:
        timeperiod = validate_rsi_params(timeperiod)
        rsi_combinations.append({
            "rsi_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not rsi_combinations:
    raise ValueError("No valid RSI parameter combinations were generated.")

adx_combinations = []
for di_length in adx_di_lengths:
    for adx_smoothing in adx_smoothing_timeperiods:
        try:
            di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
            adx_combinations.append({
                "adx_di_length": di_length,
                "adx_smoothing_timeperiod": adx_smoothing,
            })
        except ValueError:
            pass

if not adx_combinations:
    raise ValueError("No valid ADX parameter combinations were generated.")

donchian_combinations = []
for timeperiod in donchian_timeperiods:
    try:
        timeperiod = validate_donchian_params(timeperiod)
        donchian_combinations.append({
            "donchian_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not donchian_combinations:
    raise ValueError("No valid DONCHIAN parameter combinations were generated.")

trix_combinations = []
for timeperiod in trix_timeperiods:
    try:
        timeperiod = validate_trix_params(timeperiod)
        trix_combinations.append({
            "trix_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not trix_combinations:
    raise ValueError("No valid TRIX parameter combinations were generated.")

vortex_combinations = []
for timeperiod in vortex_timeperiods:
    try:
        timeperiod = validate_vortex_params(timeperiod)
        vortex_combinations.append({
            "vortex_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not vortex_combinations:
    raise ValueError("No valid VORTEX parameter combinations were generated.")

alma_combinations = []
for fast_length in alma_fast_lengths:
    for slow_length in alma_slow_lengths:
        for offset in alma_offsets:
            for sigma in alma_sigmas:
                try:
                    fast_v, slow_v, offset_v, sigma_v = validate_alma_crossover_params(
                        fast_length, slow_length, offset, sigma
                    )
                    alma_combinations.append({
                        "alma_fast_length": fast_v,
                        "alma_slow_length": slow_v,
                        "alma_offset": offset_v,
                        "alma_sigma": sigma_v,
                    })
                except ValueError:
                    pass

if not alma_combinations:
    raise ValueError("No valid ALMA parameter combinations were generated.")

indicator_grids = {
    "Triple EMA": {
        "description": "Triple EMA crossover strategy",
        "params": ema_combinations,
        "build_signals": build_triple_ema_signals,
        "format_params": format_triple_ema_params
    },
    "MACD": {
        "description": "MACD line crossing its signal line",
        "params": macd_combinations,
        "build_signals": build_macd_signals,
        "format_params": format_macd_params
    },
    "AROON": {
        "description": "AROON Up/Down crossover: buy when Up crosses above Down and Up > entry level, sell when Down crosses above Up",
        "params": aroon_combinations,
        "build_signals": build_aroon_signals,
        "format_params": format_aroon_params
    },
    "STC": {
        "description": "Schaff Trend Cycle: buy on cross above 50, sell on cross below 50",
        "params": stc_combinations,
        "build_signals": build_stc_signals,
        "format_params": format_stc_params
    },
    "KAMA": {
        "description": "Price crossing KAMA line: buy on cross above, sell on cross below",
        "params": kama_combinations,
        "build_signals": build_kama_signals,
        "format_params": format_kama_params
    },
    "SUPERTREND": {
        "description": "SUPERTREND flip: buy when close above red line, sell when close below green line",
        "params": supertrend_combinations,
        "build_signals": build_supertrend_signals,
        "format_params": format_supertrend_params
    },
    "KALMAN": {
        "description": "Price crossing Kalman filter: buy on cross above, sell on cross below",
        "params": kalman_combinations,
        "build_signals": build_kalman_signals,
        "format_params": format_kalman_params
    },
    "RSI": {
        "description": "RSI crossing 50: buy on cross above, sell on cross below",
        "params": rsi_combinations,
        "build_signals": build_rsi_signals,
        "format_params": format_rsi_params
    },
    "ADX": {
        "description": "ADX +DI/-DI crossover: buy when +DI crosses above -DI with ADX > trend level, sell when -DI crosses above +DI with ADX > trend level",
        "params": adx_combinations,
        "build_signals": build_adx_signals,
        "format_params": format_adx_params
    },
    "DONCHIAN": {
        "description": "DONCHIAN breakout: buy when close crosses above prior upper band, sell when close crosses below prior middle band",
        "params": donchian_combinations,
        "build_signals": build_donchian_signals,
        "format_params": format_donchian_params
    },
    "TRIX": {
        "description": "TRIX crossing 0: buy on cross above, sell on cross below",
        "params": trix_combinations,
        "build_signals": build_trix_signals,
        "format_params": format_trix_params
    },
    "VORTEX": {
        "description": "VORTEX VI+/VI- crossover: buy when VI+ crosses above VI-, sell when VI- crosses above VI+",
        "params": vortex_combinations,
        "build_signals": build_vortex_signals,
        "format_params": format_vortex_params
    },
    "ALMA": {
        "description": "ALMA fast/slow crossover: buy when fast crosses above slow, sell when fast crosses below slow",
        "params": alma_combinations,
        "build_signals": build_alma_signals,
        "format_params": format_alma_params
    },
}

active_indicator_names = ["Triple EMA", "MACD", "AROON", "STC", "KAMA", "SUPERTREND", "KALMAN", "RSI", "ADX", "DONCHIAN", "TRIX", "VORTEX", "ALMA"]
missing_indicator_names = [
    name for name in active_indicator_names
    if name not in indicator_grids
]
if missing_indicator_names:
    raise KeyError(f"Active indicators are not defined: {missing_indicator_names}")

active_indicator_grids = {
    name: indicator_grids[name]
    for name in active_indicator_names
}


def get_active_indicator_spec(indicator_name):
    if indicator_name not in active_indicator_grids:
        available = ", ".join(active_indicator_grids.keys())
        raise KeyError(f"Unknown indicator '{indicator_name}'. Available indicators: {available}")
    return active_indicator_grids[indicator_name]


def require_params_dict(params, indicator_name):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for {indicator_name}, got {type(params).__name__}.")
    return params




INIT_CASH = 100_000
FEE_RATE = 0.0005
SLIPPAGE_RATE = 0.0005


def trade_stats_from_portfolio(pf):
    """Extract trade-level stats; safe when there are zero trades."""
    trades = pf.trades
    total_trades = len(trades)
    stats = {
        "total_trades": total_trades,
        "win_rate_pct": np.nan,
        "profit_factor": np.nan,
        "expectancy": np.nan,
        "avg_win": np.nan,
        "avg_loss": np.nan,
    }
    if total_trades <= 0:
        return stats

    tr = trades.returns.values if hasattr(trades.returns, "values") else np.asarray(trades.returns)
    if tr.size == 0:
        return stats

    pos = tr[tr > 0]
    neg = tr[tr < 0]
    stats["win_rate_pct"] = (len(pos) / len(tr)) * 100.0
    gains = float(pos.sum()) if len(pos) else 0.0
    losses = float(abs(neg.sum())) if len(neg) else 0.0
    stats["profit_factor"] = (gains / losses) if losses > 0 else (np.inf if gains > 0 else np.nan)
    stats["expectancy"] = float(tr.mean())
    stats["avg_win"] = float(pos.mean()) if len(pos) else np.nan
    stats["avg_loss"] = float(abs(neg.mean())) if len(neg) else np.nan
    return stats


def run_signal_backtest(price, entries, exits):
    """Shared Portfolio.from_signals wrapper used by the ensemble cell (cell 7)."""
    close_arg = price.to_numpy(dtype=float) if hasattr(price, "to_numpy") else np.asarray(price, dtype=float)
    if hasattr(entries, "to_numpy"):
        entries = entries.to_numpy(dtype=bool)
    if hasattr(exits, "to_numpy"):
        exits = exits.to_numpy(dtype=bool)
    return vbt.Portfolio.from_signals(
        close=close_arg,
        entries=entries,
        exits=exits,
        init_cash=INIT_CASH,
        fees=FEE_RATE,
        slippage=SLIPPAGE_RATE,
        freq=FREQ,
    )


def fmt_metric(value, spec=".3f", suffix=""):
    """Format metrics without crashing on NaN/None/inf."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "n/a"
    try:
        return format(float(value), spec) + suffix
    except (TypeError, ValueError):
        return "n/a"


def safe_float(value, default=np.nan):
    """Convert to float; return default for missing/invalid/non-finite values."""
    try:
        out = float(value)
        return out if np.isfinite(out) else default
    except (TypeError, ValueError):
        return default


_tied_is_sharpe_notes_printed = set()


def displayed_is_top_n(indicator_results, top_n=5):
    """Top-N rows shown in the in-sample results table."""
    if "sharpe_ratio" not in indicator_results.columns:
        raise KeyError("indicator_results is missing required column: sharpe_ratio")
    return indicator_results.dropna(subset=["sharpe_ratio"]).nlargest(top_n, "sharpe_ratio")


def is_sharpe_tied_at_best(indicator_results, rtol=1e-9, atol=1e-12):
    valid = indicator_results.dropna(subset=["sharpe_ratio"])
    if valid.empty:
        return False
    best_sharpe = valid["sharpe_ratio"].max()
    tied_count = np.isclose(valid["sharpe_ratio"], best_sharpe, rtol=rtol, atol=atol).sum()
    return int(tied_count) > 1


def maybe_print_tied_is_sharpe_note(indicator_name, indicator_results, used_param_displays, context, top_n=5):
    if indicator_name in _tied_is_sharpe_notes_printed:
        return

    required_columns = {"sharpe_ratio", "param_display"}
    missing_columns = required_columns.difference(indicator_results.columns)
    if missing_columns:
        return

    used_labels = {
        str(label).strip()
        for label in used_param_displays
        if label is not None and pd.notna(label) and str(label).strip() and str(label).strip().lower() != "nan"
    }
    if not used_labels:
        return

    display_top = displayed_is_top_n(indicator_results, top_n)
    if display_top.empty:
        return

    display_labels = {
        str(label).strip()
        for label in display_top["param_display"].tolist()
        if label is not None and pd.notna(label) and str(label).strip() and str(label).strip().lower() != "nan"
    }
    outside_display = sorted(used_labels - display_labels)
    if not outside_display or not is_sharpe_tied_at_best(indicator_results):
        return

    best_sharpe = float(display_top["sharpe_ratio"].iloc[0])
    print(
        f"Note ({context}): {indicator_name} has multiple parameter sets tied at the "
        f"best in-sample Sharpe ({best_sharpe:.3f}). The in-sample top-{top_n} table shows one "
        f"valid tied set; this step used different tied equivalent(s): "
        f"{', '.join(outside_display)}"
    )
    _tied_is_sharpe_notes_printed.add(indicator_name)


def best_is_row(indicator_results):
    """Pick the single best in-sample row by Sharpe (matches the IS results table)."""
    ranked = displayed_is_top_n(indicator_results, 1)
    if ranked.empty:
        return None
    return ranked.iloc[0]


def best_oos_row(oos_results_df):
    """Pick the single best OOS row by Sharpe among validated IS top-N candidates."""
    valid = oos_results_df.dropna(subset=["OOS_Sharpe"])
    if valid.empty:
        return None
    return valid.loc[valid["OOS_Sharpe"].idxmax()]


total_grid_combinations = sum(len(spec["params"]) for spec in active_indicator_grids.values())

print("Active indicator grids:")
for indicator_name, spec in active_indicator_grids.items():
    combinations = spec["params"]
    print(f"\n{indicator_name}: {spec['description']}")
    print(f"Generated {len(combinations)} valid {indicator_name} combinations")
    print("First 10 combinations preview:")
    for i, params in enumerate(combinations[:10], 1):
        print(f"  {i:2d}. {spec['format_params'](params)}")
    if len(combinations) > 10:
        print(f"   ... and {len(combinations) - 10} more combinations")

print(f"\nTotal combinations across active grids: {total_grid_combinations}")
print("Ready to test each indicator grid separately on training data!")


Active indicator grids:

Triple EMA: Triple EMA crossover strategy
Generated 2400 valid Triple EMA combinations
First 10 combinations preview:
   1. EMA(4,90,120)
   2. EMA(4,90,121)
   3. EMA(4,90,122)
   4. EMA(4,90,123)
   5. EMA(4,90,124)
   6. EMA(4,90,125)
   7. EMA(4,90,126)
   8. EMA(4,90,127)
   9. EMA(4,90,128)
  10. EMA(4,90,129)
   ... and 2390 more combinations

MACD: MACD line crossing its signal line
Generated 128000 valid MACD combinations
First 10 combinations preview:
   1. MACD(10,60,40)
   2. MACD(10,60,41)
   3. MACD(10,60,42)
   4. MACD(10,60,43)
   5. MACD(10,60,44)
   6. MACD(10,60,45)
   7. MACD(10,60,46)
   8. MACD(10,60,47)
   9. MACD(10,60,48)
  10. MACD(10,60,49)
   ... and 127990 more combinations

AROON: AROON Up/Down crossover: buy when Up crosses above Down and Up > entry level, sell when Down crosses above Up
Generated 76 valid AROON combinations
First 10 combinations preview:
   1. AROON(4)
   2. AROON(5)
   3. AROON(6)
   4. AROON(7)
   5. AROON(8)
 

ENSEMBLE GRID SEARCH - OR COMBINATIONS WITH BORUTA (BEST PARAMS FROM RESULTS CSV)
---------------------------------------------------------------------------------

This section builds ensembles from indicators that already passed the single-indicator sweep.

- Runs separately for **each ticker** in `TICKERS` (loaded in the download cell).
- Best parameter settings are loaded from each ticker's saved `Indicator_sweep_results/{ticker_slug}_indicator_sweep_results.csv` file (no re-run of the per-indicator grids).
- Only indicators that clear the editable OOS Sharpe pass threshold (`ENSEMBLE_MIN_OOS_SHARPE`) are eligible.
- Ensembles use at most 3 indicators.
- Entry and exit signals are combined with logical OR (any member entry triggers an entry; any member exit triggers an exit).
- Each ensemble is backtested on that ticker's **training (IS)** Close series during the grid search.
- Ensembles are **optimized / ranked by in-sample Sharpe**.
- Ensembles with IS Sharpe > `ENSEMBLE_BORUTA_MIN_IS_SHARPE` receive **Boruta** validation: OOS returns are shuffled `ENSEMBLE_BORUTA_SHUFFLES` times with seed `ENSEMBLE_BORUTA_RANDOM_SEED`; `Boruta_Score` is the % of shuffles beaten by the real OOS Sharpe.
- Selected ensembles are also backtested on that ticker's **validation (OOS)** Close series for OOS metrics.
- Output: a master table of each ticker's best ensemble, then the full ensemble list table per ticker.

---



In [6]:
# ENSEMBLE GRID SEARCH (OR) + BORUTA — BEST PARAMS FROM RESULTS CSV

from itertools import combinations
import ast
import os
import re

INDICATOR_SWEEP_RESULTS_DIR = "Indicator_sweep_results"
_WINDOWS_RESERVED_NAMES = {
    "con", "prn", "aux", "nul",
    "com1", "com2", "com3", "com4", "com5", "com6", "com7", "com8", "com9",
    "lpt1", "lpt2", "lpt3", "lpt4", "lpt5", "lpt6", "lpt7", "lpt8", "lpt9",
}
_ensemble_results_path_cache = {}

# ---------------------------------------------------------------------------
# Editable pass thresholds / ensemble size limits / Boruta settings
# ---------------------------------------------------------------------------
ENSEMBLE_MIN_OOS_SHARPE = 0.7  # indicator must have OOS Sharpe > this to be eligible
ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE = 1.0  # master leaderboard: walk IS ranks until OOS Sharpe >= this
ENSEMBLE_MIN_INDICATORS = 2   # smallest ensemble size searched
ENSEMBLE_MAX_INDICATORS = 3   # largest ensemble size searched (max 3 per requirements)
# Boruta validation on top IS ensembles (custom Boruta shuffle test)
ENSEMBLE_BORUTA_MIN_IS_SHARPE = 0.95  # ensembles with IS Sharpe > this sent to Boruta / OOS metrics
ENSEMBLE_BORUTA_SHUFFLES = 50    # OOS return permutation count
ENSEMBLE_BORUTA_MIN_OOS_BARS = 11  # require at least this many finite OOS return bars
ENSEMBLE_BORUTA_RANDOM_SEED = 42  # RNG seed so Boruta scores are reproducible


def _parse_sharpe_bound(value, prefer="max"):
    """Parse a CSV sharpe cell ('1.248', '0.715-0.728', 'n/a') to a float bound.

    Kept for legacy range-summary CSVs; per-row exports use numeric columns directly.
    """
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return np.nan
    text = str(value).strip()
    if not text or text.lower() in {"n/a", "nan", "none"}:
        return np.nan
    if re.search(r"\bto\b", text, flags=re.IGNORECASE):
        parts = re.split(r"\s+to\s+", text, flags=re.IGNORECASE)
    else:
        parts = re.split(r"(?<=\d)-(?=\d)", text)
    parsed = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        try:
            parsed.append(float(part))
        except ValueError:
            numbers = re.findall(
                r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?",
                part,
            )
            parsed.extend(float(x) for x in numbers)
    if not parsed:
        return np.nan
    prefer_norm = str(prefer).strip().lower()
    if prefer_norm not in {"min", "max"}:
        raise ValueError(f"prefer must be 'min' or 'max'; got {prefer!r}")
    if prefer_norm == "min":
        return float(min(parsed))
    return float(max(parsed))


def _finite_sharpe_value(value, prefer="max"):
    """Parse a numeric or legacy range Sharpe cell to one finite float."""
    number = safe_float(value)
    if np.isfinite(number):
        return float(number)
    return _parse_sharpe_bound(value, prefer=prefer)


def _parse_param_tokens(param_range):
    """Extract the inside of Name(...) from a legacy param_range label."""
    text = str(param_range).strip()
    match = re.search(r"\((.*)\)\s*$", text)
    if not match:
        raise ValueError(f"Could not parse param_range label: {param_range!r}")
    inside = match.group(1).strip()
    if not inside:
        raise ValueError(f"Empty parameter list in param_range label: {param_range!r}")
    return [token.strip() for token in inside.split(",")]


# Local copy of param keys so CSV loading does not need a separate export step.
_ENSEMBLE_PARAM_KEYS = {
    "Triple EMA": ("ema1_period", "ema2_period", "ema3_period"),
    "MACD": ("macd_fast_period", "macd_slow_period", "macd_signal_period"),
    "AROON": ("aroon_timeperiod",),
    "STC": ("stc_ema_short_period", "stc_ema_long_period", "stc_cycle_period"),
    "KAMA": ("kama_period", "kama_fast_ema_constant", "kama_slow_ema_constant"),
    "SUPERTREND": ("supertrend_atr_length", "supertrend_factor"),
    "KALMAN": ("kalman_process_noise", "kalman_measurement_noise"),
    "RSI": ("rsi_timeperiod",),
    "ADX": ("adx_di_length", "adx_smoothing_timeperiod"),
    "DONCHIAN": ("donchian_timeperiod",),
    "TRIX": ("trix_timeperiod",),
    "VORTEX": ("vortex_timeperiod",),
    "ALMA": ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma"),
}

# Named CSV forms: SUPERTREND(ATR=36, Factor=4), KALMAN(Q=1, R=113), ADX(DI=19, Smooth=44)
_ENSEMBLE_NAMED_ALIASES = {
    "SUPERTREND": ("atr", "factor"),
    "KALMAN": ("q", "r"),
    "ADX": ("di", "smooth"),
}


def _normalize_params_dict(params):
    """Normalize param dicts for equality checks (numpy scalars, int-like floats)."""
    if not isinstance(params, dict):
        return params
    normalized = {}
    for key, value in params.items():
        if isinstance(value, np.generic):
            value = value.item()
        if isinstance(value, float) and np.isfinite(value) and value == int(value):
            value = int(value)
        normalized[str(key)] = value
    return normalized


def _parse_params_from_full_param_range(indicator_name, param_range):
    """Rebuild a params dict from a legacy full-stage CSV param_range label."""
    keys = _ENSEMBLE_PARAM_KEYS.get(indicator_name)
    if not keys:
        raise KeyError(f"No param keys registered for indicator {indicator_name!r}")

    tokens = _parse_param_tokens(param_range)
    named = {}
    positional = []
    for token in tokens:
        if "=" in token:
            name, raw = token.split("=", 1)
            named[name.strip().lower()] = raw.strip()
        else:
            positional.append(token)

    if named and positional:
        raise ValueError(
            f"{indicator_name}: mixed named/positional params in {param_range!r}"
        )

    if named:
        aliases = _ENSEMBLE_NAMED_ALIASES.get(indicator_name)
        if aliases is None or len(aliases) != len(keys):
            raise ValueError(
                f"{indicator_name}: unsupported named param_range format: {param_range!r}"
            )
        values = []
        for alias in aliases:
            if alias not in named:
                raise KeyError(
                    f"{indicator_name}: missing '{alias}=' in param_range {param_range!r}"
                )
            values.append(named[alias])
    else:
        if len(positional) != len(keys):
            raise ValueError(
                f"{indicator_name}: expected {len(keys)} values in {param_range!r}, "
                f"got {len(positional)}"
            )
        values = positional

    params = {}
    for key, raw in zip(keys, values):
        number = float(raw)
        if key == "alma_offset":
            # CSV displays fractional offset (0.14); signal builder stores hundredths (14).
            if abs(number) <= 1.0:
                params[key] = int(round(number * 100.0))
            else:
                params[key] = int(round(number))
        elif number == int(number):
            params[key] = int(number)
        else:
            params[key] = float(number)
    return params


def _coerce_params_cell(value, indicator_name):
    """Parse a per-row CSV params cell into a dict."""
    if isinstance(value, dict):
        return _normalize_params_dict(value)
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        raise TypeError(f"{indicator_name}: params cell is missing")
    if not isinstance(value, str):
        raise TypeError(
            f"{indicator_name}: expected params dict/string, got {type(value).__name__}"
        )
    text = value.strip()
    if not (text.startswith("{") and text.endswith("}")):
        raise TypeError(f"{indicator_name}: params cell is not a dict literal: {value!r}")
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError, MemoryError) as exc:
        raise TypeError(f"{indicator_name}: could not parse params cell: {value!r}") from exc
    if not isinstance(parsed, dict):
        raise TypeError(
            f"{indicator_name}: params cell parsed to {type(parsed).__name__}, not dict"
        )
    return _normalize_params_dict(parsed)


def _require_indicator_param_keys(indicator_name, params):
    """Ensure parsed params include the registered keys for this indicator."""
    keys = _ENSEMBLE_PARAM_KEYS.get(indicator_name)
    if not keys:
        raise KeyError(f"No param keys registered for indicator {indicator_name!r}")
    if not isinstance(params, dict):
        raise TypeError(
            f"{indicator_name}: expected params dict, got {type(params).__name__}"
        )
    missing = [key for key in keys if key not in params]
    if missing:
        raise KeyError(f"{indicator_name}: params missing required keys: {missing}")
    return params


def _params_from_full_row(indicator_name, row):
    """Prefer per-row `params`; fall back to legacy `param_range` labels."""
    params = None
    if "params" in row.index and pd.notna(row.get("params")):
        text = str(row.get("params")).strip()
        if text and text.lower() not in {"n/a", "nan", "none"}:
            params = _coerce_params_cell(row.get("params"), indicator_name)
    if params is None and "param_range" in row.index and pd.notna(row.get("param_range")):
        params = _parse_params_from_full_param_range(indicator_name, row.get("param_range"))
    if params is None:
        raise KeyError(
            f"{indicator_name}: full-stage row needs a `params` or `param_range` value"
        )
    return _require_indicator_param_keys(indicator_name, params)


def _metric_column(frame, candidates):
    """Return the first present metric column name from candidates."""
    for col in candidates:
        if col in frame.columns:
            return col
    return None


def _max_sharpe_by_indicator(frame, metric_candidates):
    """Max finite Sharpe per indicator across (possibly many) stage rows."""
    if frame is None or frame.empty:
        return {}
    col = _metric_column(frame, metric_candidates)
    if col is None:
        col = _metric_column(frame, ("sharpe",))
    if col is None:
        return {}
    metrics = frame[col].map(lambda value: _finite_sharpe_value(value, prefer="max"))
    tmp = pd.DataFrame({
        "indicator": frame["indicator"].astype(str),
        "metric": pd.to_numeric(metrics, errors="coerce"),
    })
    out = {}
    for indicator_name, grp in tmp.groupby("indicator", sort=False):
        finite = grp["metric"].dropna()
        finite = finite[np.isfinite(finite)]
        out[str(indicator_name)] = float(finite.max()) if not finite.empty else np.nan
    return out


def _sharpe_matching_params(frame, indicator_name, params, metric_candidates):
    """Return Sharpe for the row whose params match, else NaN."""
    if frame is None or frame.empty or params is None:
        return np.nan
    ind_mask = frame["indicator"].astype(str) == str(indicator_name)
    subset = frame.loc[ind_mask]
    if subset.empty or "params" not in subset.columns:
        return np.nan
    col = _metric_column(subset, metric_candidates)
    if col is None:
        col = _metric_column(subset, ("sharpe",))
    if col is None:
        return np.nan
    target = _normalize_params_dict(params)
    for _, row in subset.iterrows():
        try:
            row_params = _coerce_params_cell(row.get("params"), indicator_name)
        except TypeError:
            continue
        if row_params == target:
            return _finite_sharpe_value(row.get(col), prefer="max")
    return np.nan


def _notebook_dir():
    """Best-effort project/notebook directory (cwd fallback in Jupyter)."""
    try:
        return os.path.dirname(os.path.abspath(__file__))
    except NameError:
        return os.getcwd()


def _ticker_results_slug(ticker):
    """Match indicator-sweep slug rules for `{slug}_indicator_sweep_results.*`."""
    text = str(ticker).strip()
    if not text:
        raise ValueError(f"Could not build results filename from TICKER={ticker!r}")
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower())
    slug = re.sub(r"_+", "_", slug).strip("_")
    if not slug:
        raise ValueError(f"Could not build results filename from TICKER={ticker!r}")
    if slug in _WINDOWS_RESERVED_NAMES:
        slug = f"asset_{slug}"
    return slug


def _indicator_sweep_results_filename(ticker, file_format="csv"):
    fmt = str(file_format).strip().lower().lstrip(".")
    if fmt not in {"csv", "parquet"}:
        raise ValueError(f"file_format must be 'csv' or 'parquet'; got {file_format!r}")
    return f"{_ticker_results_slug(ticker)}_indicator_sweep_results.{fmt}"


def _indicator_sweep_results_search_dirs():
    """Directories to search for sweep CSVs (notebook dir, cwd, and subfolders)."""
    dirs = []
    for base in (_notebook_dir(), os.getcwd()):
        base = os.path.abspath(base)
        if base not in dirs:
            dirs.append(base)
        subdir = os.path.join(base, INDICATOR_SWEEP_RESULTS_DIR)
        if subdir not in dirs:
            dirs.append(subdir)
    return dirs


def _resolve_existing_results_path(filename):
    """Return the first existing regular file named `filename` in search dirs."""
    for directory in _indicator_sweep_results_search_dirs():
        candidate = os.path.join(directory, filename)
        if os.path.isfile(candidate):
            return candidate
    return None


def _resolve_ensemble_results_filepath(ticker=None):
    """Resolve `{ticker_slug}_indicator_sweep_results.csv` for a ticker.

    Never silently falls back to another ticker's CSV.
    """
    if ticker is None:
        if "TICKER" not in globals() or not str(TICKER).strip():
            raise ValueError(
                "ticker is required (or set TICKER in cell 2) to resolve the "
                "indicator sweep results CSV."
            )
        ticker = TICKER
    ticker = str(ticker).strip()
    if not ticker:
        raise ValueError("ticker is empty. Provide a non-empty ticker symbol.")

    cache_key = ticker.upper()
    if cache_key in _ensemble_results_path_cache:
        return _ensemble_results_path_cache[cache_key]

    filename = None
    if "_results_filepath_for_ticker" in globals():
        try:
            helper_path = _results_filepath_for_ticker(ticker, file_format="csv")
        except Exception as exc:
            raise FileNotFoundError(
                f"Could not resolve indicator sweep results CSV for TICKER={ticker!r}: "
                f"_results_filepath_for_ticker failed: {exc}"
            ) from exc
        if helper_path and str(helper_path).strip():
            helper_path = str(helper_path).strip()
            if os.path.isfile(helper_path):
                _ensemble_results_path_cache[cache_key] = helper_path
                return helper_path
            filename = os.path.basename(helper_path)

    if not filename:
        filename = _indicator_sweep_results_filename(ticker, file_format="csv")

    resolved = _resolve_existing_results_path(filename)
    if resolved:
        _ensemble_results_path_cache[cache_key] = resolved
        return resolved

    searched = [os.path.join(directory, filename) for directory in _indicator_sweep_results_search_dirs()]
    raise FileNotFoundError(
        f"Could not resolve indicator sweep results CSV for TICKER={ticker!r}. "
        f"Expected filename {filename!r}. Looked in: {searched!r}"
    )


def _close_series_for_ticker(df, ticker):
    """Select Close for `ticker` only — never fall back to another symbol's Close."""
    ticker = str(ticker).strip()
    if not ticker:
        raise ValueError("ticker is empty when selecting Close series.")
    if df is None or getattr(df, "empty", True):
        raise ValueError(
            f"stock_data is missing or empty when selecting Close for {ticker!r}."
        )

    if isinstance(df.columns, pd.MultiIndex):
        if ("Close", ticker) in df.columns:
            series = df[("Close", ticker)]
        elif (ticker, "Close") in df.columns:
            series = df[(ticker, "Close")]
        else:
            raise KeyError(
                f"Close not found for ticker {ticker!r} in stock_data MultiIndex columns."
            )
    else:
        if "Close" not in df.columns:
            raise KeyError("Close not found in stock_data columns.")
        # Flat single-asset frame is only valid when searching that one symbol.
        series = df["Close"]

    series = pd.Series(series.astype(float).to_numpy(), index=series.index, name="price")
    series = series.replace([np.inf, -np.inf], np.nan).dropna()
    if series.empty or len(series) < 2:
        raise ValueError(
            f"Close series for TICKER={ticker!r} is missing or too short after cleaning."
        )
    return series


def _lookup_stage_sharpe(stage_map, indicator_name):
    """Return stage Sharpe for an indicator, or NaN if missing."""
    if indicator_name not in stage_map:
        return np.nan
    return stage_map[indicator_name]


def _resolve_pass_sharpes(indicator_name, csv_is_sharpe, csv_oos_sharpe, csv_params=None):
    """Prefer exact IS/OOS from best_by_indicator; else CSV values.

    Returns (is_sharpe, oos_sharpe, source_label, oos_from_csv).
    oos_from_csv is True when the OOS value used for eligibility came from CSV
    (including param-mismatch fallback), not from best_by_indicator.
    """
    is_sharpe = csv_is_sharpe
    oos_sharpe = csv_oos_sharpe
    used_stored_is = False
    used_stored_oos = False

    if "best_by_indicator" in globals() and isinstance(best_by_indicator, dict):
        stored = best_by_indicator.get(indicator_name)
        if isinstance(stored, dict):
            stored_params = stored.get("params")
            if (
                stored_params is not None
                and csv_params is not None
                and _normalize_params_dict(stored_params) != _normalize_params_dict(csv_params)
            ):
                # Params disagree: do not trust stored Sharpes for this CSV strategy.
                return is_sharpe, oos_sharpe, "csv_params_mismatch", True

            stored_is = safe_float(stored.get("is_sharpe"))
            stored_oos = safe_float(stored.get("oos_sharpe"))
            if np.isfinite(stored_is):
                is_sharpe = stored_is
                used_stored_is = True
            if np.isfinite(stored_oos):
                oos_sharpe = stored_oos
                used_stored_oos = True

    oos_from_csv = not used_stored_oos
    if used_stored_is and used_stored_oos:
        source = "best_by_indicator"
    elif used_stored_oos and not used_stored_is:
        source = "best_by_indicator+csv_is"
    elif used_stored_is and not used_stored_oos:
        source = "csv_oos+best_is"
    else:
        source = "csv"
    return is_sharpe, oos_sharpe, source, oos_from_csv

def _or_combine_member_signals(signal_map, members):
    first_entries, first_exits = signal_map[members[0]]
    entries_or = first_entries
    exits_or = first_exits
    for name in members[1:]:
        entries_i, exits_i = signal_map[name]
        entries_or = entries_or | entries_i
        exits_or = exits_or | exits_i
    return entries_or, exits_or


def _backtest_ensemble(price, signal_map, members, years, need_returns=False):
    missing = [name for name in members if name not in signal_map]
    if missing:
        raise KeyError(f"missing member signals for: {missing}")
    entries_or, exits_or = _or_combine_member_signals(signal_map, members)
    pf = run_signal_backtest(price, entries_or, exits_or)
    sharpe = safe_float(pf.sharpe_ratio(freq=FREQ, year_freq=YEAR_FREQ))
    sortino = safe_float(pf.sortino_ratio(freq=FREQ, year_freq=YEAR_FREQ))
    total_return = safe_float(pf.total_return())
    max_dd = safe_float(pf.max_drawdown())
    trade_stats = trade_stats_from_portfolio(pf)
    total_trades = int(trade_stats.get("total_trades") or 0)
    trades_per_year = total_trades / years if np.isfinite(years) and years > 0 else np.nan
    result = {
        "sharpe": sharpe,
        "sortino": sortino,
        "total_return": total_return,
        "max_drawdown": max_dd,
        "total_trades": total_trades,
        "win_rate": trade_stats["win_rate_pct"],
        "profit_factor": trade_stats["profit_factor"],
        "trades_per_year": trades_per_year,
        "returns": None,
    }
    if need_returns:
        returns = pf.returns()
        if not isinstance(returns, pd.Series):
            returns = pd.Series(np.asarray(returns, dtype=float))
        result["returns"] = returns
    return result


# ---------------------------------------------------------------------------
# Boruta-with-Sharpe (shadow LOO delta-Sharpe + binomial decisions)
# ---------------------------------------------------------------------------
_BORUTA_CONFIRMED, _BORUTA_TENTATIVE, _BORUTA_REJECTED = (
    "Confirmed",
    "Tentative",
    "Rejected",
)


def _as_bool1d(signal, label):
    """Coerce a signal to a 1-D bool ndarray."""
    arr = np.asarray(signal, dtype=bool)
    if arr.ndim != 1:
        raise ValueError(f"{label} must be 1-D; got shape {arr.shape}")
    return arr


def _make_shadow(entries, exits, rng, min_shift_frac=0.05):
    """Circular block shift: keep signal structure, destroy price alignment."""
    entries = _as_bool1d(entries, "shadow entries")
    exits = _as_bool1d(exits, "shadow exits")
    if entries.size == 0 or exits.size == 0:
        raise ValueError("shadow signals must be non-empty")
    if entries.size != exits.size:
        raise ValueError(
            f"shadow entries/exits length mismatch: {entries.size} vs {exits.size}"
        )
    n = int(entries.size)
    lo = max(1, int(n * min_shift_frac))
    k = int(rng.integers(lo, n - lo)) if n > 2 * lo else 1
    return np.roll(entries, k), np.roll(exits, k)


def _vote_combine_member_signals(signal_map, members, k=None):
    """k-of-n vote. Degrades when a member is noise; pure OR does not."""
    members = list(members)
    if not members:
        raise ValueError("vote combine needs >= 1 member")
    missing = [name for name in members if name not in signal_map]
    if missing:
        raise KeyError(f"missing member signals for: {missing}")
    k = k if k is not None else max(1, (len(members) + 1) // 2)

    first_entries, first_exits = signal_map[members[0]]
    first_entries = _as_bool1d(first_entries, f"{members[0]} entries")
    first_exits = _as_bool1d(first_exits, f"{members[0]} exits")
    if first_entries.size == 0 or first_exits.size == 0:
        raise ValueError(f"{members[0]} signals must be non-empty")
    if first_entries.size != first_exits.size:
        raise ValueError(
            f"{members[0]} entries/exits length mismatch: "
            f"{first_entries.size} vs {first_exits.size}"
        )

    n = int(first_entries.size)
    entry_counts = np.zeros(n, dtype=int)
    exit_counts = np.zeros(n, dtype=int)
    for name in members:
        entries_i, exits_i = signal_map[name]
        entries_i = _as_bool1d(entries_i, f"{name} entries")
        exits_i = _as_bool1d(exits_i, f"{name} exits")
        if entries_i.size != n or exits_i.size != n:
            raise ValueError(
                f"{name} signal length mismatch: entries={entries_i.size}, "
                f"exits={exits_i.size}, expected={n}"
            )
        entry_counts += entries_i
        exit_counts += exits_i
    return entry_counts >= k, exit_counts >= k


def _finite_delta(left, right):
    """Return left-right only when both sides are finite; else NaN."""
    try:
        left_f = float(left)
        right_f = float(right)
    except (TypeError, ValueError):
        return np.nan
    if not (np.isfinite(left_f) and np.isfinite(right_f)):
        return np.nan
    return left_f - right_f


def _boruta_sharpe(
    sig_map,
    sharpe_fn,
    n_iter=100,
    alpha=0.05,
    seed=42,
    combine="vote",
    max_rounds=10,
    verbose=False,
):
    """Run Boruta on indicator signals ranked by leave-one-out delta Sharpe.

    sig_map   : {name: (entries, exits)} — boolean arrays/Series, same length
    sharpe_fn : (entries, exits) -> float
    returns   : dict {name: (decision, hits, n_iter, mean_importance)}
    """
    if not callable(sharpe_fn):
        raise TypeError("sharpe_fn must be callable")
    if combine not in {"or", "vote"}:
        raise ValueError(f"combine must be 'or' or 'vote'; got {combine!r}")
    comb = (
        _or_combine_member_signals
        if combine == "or"
        else _vote_combine_member_signals
    )
    rng = np.random.default_rng(seed)
    pool = list(sig_map)
    if len(pool) < 2:
        raise ValueError("need >= 2 members")

    # Validate the full pool once up front (lengths / emptiness).
    _vote_combine_member_signals(sig_map, pool)

    decision = {name: _BORUTA_TENTATIVE for name in pool}
    hits = {name: 0 for name in pool}
    iters_run = {name: 0 for name in pool}
    imp_hist = {name: [] for name in pool}

    for rnd in range(max_rounds):
        active = [name for name in pool if decision[name] == _BORUTA_TENTATIVE]
        if len(active) < 2:
            break

        for _ in range(n_iter):
            work = dict(sig_map)
            # Full-ensemble Sharpe depends only on `active` — compute once.
            s_full = sharpe_fn(*comb(work, active))
            imp_real, imp_shadow = {}, {}
            for member in active:
                rest = [name for name in active if name != member]
                if not rest:
                    continue
                s_rest = sharpe_fn(*comb(work, rest))
                imp_real[member] = _finite_delta(s_full, s_rest)

                shadow_name = f"__shadow__{member}"
                work[shadow_name] = _make_shadow(*sig_map[member], rng=rng)
                s_shadow = sharpe_fn(*comb(work, rest + [shadow_name]))
                imp_shadow[member] = _finite_delta(s_shadow, s_rest)
                del work[shadow_name]

            if not imp_shadow:
                break
            finite_shadows = [
                value for value in imp_shadow.values() if np.isfinite(value)
            ]
            if not finite_shadows:
                # No usable null — still count the iteration, but no hits.
                for member, value in imp_real.items():
                    iters_run[member] += 1
                    imp_hist[member].append(value)
                continue
            shadow_max = max(finite_shadows)
            for member, value in imp_real.items():
                iters_run[member] += 1
                imp_hist[member].append(value)
                if np.isfinite(value) and value > shadow_max:
                    hits[member] += 1

        # Binomial decisions, Bonferroni over still-undecided members.
        m = len(active)
        a = alpha / max(m, 1)
        for member in active:
            t = iters_run[member]
            if t == 0:
                continue
            p_hi = stats.binomtest(hits[member], t, 0.5, alternative="greater").pvalue
            p_lo = stats.binomtest(hits[member], t, 0.5, alternative="less").pvalue
            if p_hi < a:
                decision[member] = _BORUTA_CONFIRMED
            elif p_lo < a:
                decision[member] = _BORUTA_REJECTED

        if verbose:
            print(
                f"round {rnd + 1}: "
                + ", ".join(
                    f"{name}={decision[name][:4]}({hits[name]}/{iters_run[name]})"
                    for name in pool
                )
            )
        if all(decision[name] != _BORUTA_TENTATIVE for name in pool):
            break
        if not any(decision[name] == _BORUTA_REJECTED for name in active):
            break  # nothing dropped -> refitting changes nothing

    return {
        name: (
            decision[name],
            hits[name],
            iters_run[name],
            float(np.mean(imp_hist[name])) if imp_hist[name] else np.nan,
        )
        for name in pool
    }


def _boruta_score_from_signals(sig_map, sharpe_fn, n_iter, seed):
    """Boruta-with-Sharpe score: mean hit-rate (%) across ensemble members."""
    if sig_map is None or sharpe_fn is None:
        return np.nan
    if not isinstance(sig_map, dict):
        return np.nan
    try:
        n_iter = int(n_iter)
        seed = int(seed)
    except (TypeError, ValueError):
        return np.nan
    if n_iter < 1 or seed < 0 or len(sig_map) < 2:
        return np.nan

    try:
        result = _boruta_sharpe(
            sig_map,
            sharpe_fn,
            n_iter=n_iter,
            seed=seed,
            combine="vote",
            verbose=False,
        )
    except (TypeError, ValueError, KeyError):
        return np.nan

    rates = []
    for _name, (_decision, hit_count, iters, _importance) in result.items():
        if iters > 0:
            rates.append(100.0 * float(hit_count) / float(iters))
    if not rates:
        return np.nan
    return float(np.mean(rates))

# ---------------------------------------------------------------------------
# Validate helpers + editable thresholds (once)
# ---------------------------------------------------------------------------
required_helpers = [
    "stock_data",
    "TRAIN_RATIO",
    "get_active_indicator_spec",
    "run_signal_backtest",
    "trade_stats_from_portfolio",
    "sample_years",
    "fmt_metric",
    "safe_float",
    "FREQ",
    "YEAR_FREQ",
    "TRADING_DAYS_PER_YEAR",
]
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise NameError(
        "Missing required helpers/data from earlier cells: "
        + ", ".join(missing_helpers)
    )

_oos_min = safe_float(ENSEMBLE_MIN_OOS_SHARPE)
if not np.isfinite(_oos_min):
    raise ValueError(
        f"ENSEMBLE_MIN_OOS_SHARPE must be a finite number; got {ENSEMBLE_MIN_OOS_SHARPE!r}"
    )
ENSEMBLE_MIN_OOS_SHARPE = float(_oos_min)

_lb_oos_min = safe_float(ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE)
if not np.isfinite(_lb_oos_min):
    raise ValueError(
        "ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE must be a finite number; "
        f"got {ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE!r}"
    )
ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE = float(_lb_oos_min)

if not isinstance(ENSEMBLE_MIN_INDICATORS, (int, np.integer)):
    raise ValueError(
        f"ENSEMBLE_MIN_INDICATORS must be an integer; got {ENSEMBLE_MIN_INDICATORS!r}"
    )
if not isinstance(ENSEMBLE_MAX_INDICATORS, (int, np.integer)):
    raise ValueError(
        f"ENSEMBLE_MAX_INDICATORS must be an integer; got {ENSEMBLE_MAX_INDICATORS!r}"
    )
ENSEMBLE_MIN_INDICATORS = int(ENSEMBLE_MIN_INDICATORS)
ENSEMBLE_MAX_INDICATORS = int(ENSEMBLE_MAX_INDICATORS)
if ENSEMBLE_MIN_INDICATORS < 2:
    raise ValueError("ENSEMBLE_MIN_INDICATORS must be >= 2 for OR ensembles.")
if ENSEMBLE_MAX_INDICATORS < ENSEMBLE_MIN_INDICATORS:
    raise ValueError("ENSEMBLE_MAX_INDICATORS must be >= ENSEMBLE_MIN_INDICATORS.")
if ENSEMBLE_MAX_INDICATORS > 3:
    raise ValueError("ENSEMBLE_MAX_INDICATORS must be <= 3 per requirements.")
_is_min = safe_float(ENSEMBLE_BORUTA_MIN_IS_SHARPE)
if not np.isfinite(_is_min):
    raise ValueError(
        f"ENSEMBLE_BORUTA_MIN_IS_SHARPE must be a finite number; got {ENSEMBLE_BORUTA_MIN_IS_SHARPE!r}"
    )
ENSEMBLE_BORUTA_MIN_IS_SHARPE = float(_is_min)
if (
    not isinstance(ENSEMBLE_BORUTA_SHUFFLES, (int, np.integer))
    or int(ENSEMBLE_BORUTA_SHUFFLES) < 1
):
    raise ValueError(
        f"ENSEMBLE_BORUTA_SHUFFLES must be an integer >= 1; got {ENSEMBLE_BORUTA_SHUFFLES!r}"
    )
ENSEMBLE_BORUTA_SHUFFLES = int(ENSEMBLE_BORUTA_SHUFFLES)
if (
    not isinstance(ENSEMBLE_BORUTA_MIN_OOS_BARS, (int, np.integer))
    or int(ENSEMBLE_BORUTA_MIN_OOS_BARS) < 1
):
    raise ValueError(
        f"ENSEMBLE_BORUTA_MIN_OOS_BARS must be an integer >= 1; "
        f"got {ENSEMBLE_BORUTA_MIN_OOS_BARS!r}"
    )
ENSEMBLE_BORUTA_MIN_OOS_BARS = int(ENSEMBLE_BORUTA_MIN_OOS_BARS)
if (
    not isinstance(ENSEMBLE_BORUTA_RANDOM_SEED, (int, np.integer))
    or int(ENSEMBLE_BORUTA_RANDOM_SEED) < 0
):
    raise ValueError(
        f"ENSEMBLE_BORUTA_RANDOM_SEED must be an integer >= 0; "
        f"got {ENSEMBLE_BORUTA_RANDOM_SEED!r}"
    )
ENSEMBLE_BORUTA_RANDOM_SEED = int(ENSEMBLE_BORUTA_RANDOM_SEED)
_tdpy = safe_float(TRADING_DAYS_PER_YEAR)
if not np.isfinite(_tdpy) or _tdpy <= 0:
    raise ValueError(
        f"TRADING_DAYS_PER_YEAR must be a finite number > 0; got {TRADING_DAYS_PER_YEAR!r}"
    )

# ---------------------------------------------------------------------------
# Multi-ticker ensemble grid search (one independent search per loaded ticker)
# ---------------------------------------------------------------------------
_train_ratio = safe_float(TRAIN_RATIO)
if not np.isfinite(_train_ratio) or not (0.0 < _train_ratio < 1.0):
    raise ValueError(
        f"TRAIN_RATIO must be a finite number strictly between 0 and 1; got {TRAIN_RATIO!r}"
    )
TRAIN_RATIO = float(_train_ratio)

if "TICKERS" in globals() and TICKERS is not None:
    if isinstance(TICKERS, str):
        raise ValueError("TICKERS must be a list of ticker symbols, not a single string.")
    if not isinstance(TICKERS, (list, tuple)):
        raise ValueError(
            f"TICKERS must be a list/tuple of ticker symbols; got {type(TICKERS).__name__}."
        )
    _ensemble_tickers = list(dict.fromkeys(str(t).strip() for t in TICKERS if str(t).strip()))
elif "TICKER" in globals() and str(TICKER).strip():
    _ensemble_tickers = [str(TICKER).strip()]
else:
    raise ValueError(
        "TICKERS/TICKER are missing. Run cell 2 (download) first so loaded tickers exist."
    )
if not _ensemble_tickers:
    raise ValueError("No tickers available for ensemble grid search.")
if stock_data is None or getattr(stock_data, "empty", True):
    raise ValueError("stock_data is missing or empty. Run cell 2 (download) first.")

_original_ticker = _ensemble_tickers[0]
_saved_best_by_indicator = (
    best_by_indicator
    if ("best_by_indicator" in globals() and isinstance(best_by_indicator, dict))
    else None
)
# In-memory best_by_indicator is from a prior single-ticker sweep. For multi-ticker
# searches the per-ticker results CSV is the only safe eligibility source.
_use_best_by_indicator = (
    len(_ensemble_tickers) == 1 and _saved_best_by_indicator is not None
)

all_ensemble_frames = []
ensemble_results_by_ticker = {}
_ensemble_ticker_failures = []

print("=" * 120)
print(f"ENROLLING ENSEMBLE SEARCH FOR {len(_ensemble_tickers)} TICKERS")
print(f"Tickers: {', '.join(_ensemble_tickers)}")
print("=" * 120)

for _ens_ticker in _ensemble_tickers:
    TICKER = str(_ens_ticker).strip()
    if not TICKER:
        _ensemble_ticker_failures.append("empty ticker symbol in TICKERS")
        continue

    if _use_best_by_indicator:
        best_by_indicator = _saved_best_by_indicator
    else:
        best_by_indicator = {}

    print("\n" + "#" * 120)
    print(f"# TICKER: {TICKER}")
    print("#" * 120)

    try:
        # Prefer splits prepared in cell 3; otherwise build from stock_data
        if (
            "close_by_ticker" in globals()
            and isinstance(close_by_ticker, dict)
            and TICKER in close_by_ticker
            and TICKER in train_close_by_ticker
            and TICKER in val_close_by_ticker
        ):
            close = close_by_ticker[TICKER]
            train_close = train_close_by_ticker[TICKER]
            val_close = val_close_by_ticker[TICKER]
        else:
            close = _close_series_for_ticker(stock_data, TICKER)
            split_idx = int(len(close) * TRAIN_RATIO)
            if split_idx < 1 or split_idx >= len(close):
                raise ValueError(
                    f"TRAIN_RATIO={TRAIN_RATIO!r} produced an invalid split for TICKER={TICKER!r} "
                    f"(len={len(close)}, split_idx={split_idx})."
                )
            train_close = close.iloc[:split_idx].copy()
            val_close = close.iloc[split_idx:].copy()
        if len(train_close) < 2:
            raise ValueError(
                f"train_close is missing or too short for TICKER={TICKER!r}."
            )
        if len(val_close) < 2:
            raise ValueError(
                f"val_close is missing or too short for TICKER={TICKER!r}."
            )

        ensemble_results_path = _resolve_ensemble_results_filepath(TICKER)

        try:
            sweep_results_df = pd.read_csv(ensemble_results_path)
        except Exception as exc:
            raise OSError(f"Failed to read results CSV {ensemble_results_path!r}: {exc}") from exc

        if sweep_results_df is None or sweep_results_df.empty:
            raise ValueError(f"Results CSV is empty: {ensemble_results_path}")

        required_cols = {"stage", "indicator"}
        missing_cols = required_cols.difference(sweep_results_df.columns)
        if missing_cols:
            raise KeyError(f"Results CSV missing required columns: {sorted(missing_cols)}")
        if ("params" not in sweep_results_df.columns) and ("param_range" not in sweep_results_df.columns):
            raise KeyError(
                "Results CSV needs `params` (per-row export) or legacy `param_range` "
                "to rebuild full-sample strategies."
            )

        stage_lower = sweep_results_df["stage"].astype(str).str.lower()
        full_rows = sweep_results_df[stage_lower == "full"].copy()
        is_rows = sweep_results_df[stage_lower == "is"].copy()
        oos_rows = sweep_results_df[stage_lower == "oos"].copy()
        if full_rows.empty:
            raise ValueError(f"No stage='full' rows found in {ensemble_results_path}")

        dup_full = full_rows["indicator"].astype(str).duplicated(keep=False)
        if dup_full.any():
            dup_names = sorted(full_rows.loc[dup_full, "indicator"].astype(str).unique())
            raise ValueError(
                f"Duplicate stage='full' rows for indicator(s): {dup_names}. "
                "Expected one full-sample best-param row per indicator."
            )

        # IS map is display-only (eligibility is OOS-only).
        is_sharpe_by_indicator = _max_sharpe_by_indicator(
            is_rows, ("sharpe_ratio", "sharpe", "IS_Sharpe")
        )
        # OOS map feeds the eligibility gate. Multiple top-N rows per indicator are expected.
        oos_sharpe_by_indicator = _max_sharpe_by_indicator(
            oos_rows, ("OOS_Sharpe", "sharpe", "oos_sharpe")
        )
        _has_best_by_indicator = (
            "best_by_indicator" in globals()
            and isinstance(best_by_indicator, dict)
            and bool(best_by_indicator)
        )
        if oos_rows.empty and "oos_sharpe" not in full_rows.columns and not _has_best_by_indicator:
            raise ValueError(
                f"No stage='oos' rows in {ensemble_results_path} and best_by_indicator "
                "is unavailable; cannot apply ENSEMBLE_MIN_OOS_SHARPE eligibility gate. "
                "Provide a results CSV that includes stage='oos' rows."
            )

        ensemble_candidates = {}
        oos_gate_used_csv = False
        oos_pass_used_csv_fallback = False
        print("ENSEMBLE ELIGIBILITY (from results CSV full-sample params)")
        print("=" * 70)
        print(f"Results file: {ensemble_results_path}")
        print(f"Pass threshold: OOS Sharpe > {ENSEMBLE_MIN_OOS_SHARPE}")
        print(f"Ensemble sizes: {ENSEMBLE_MIN_INDICATORS} to {ENSEMBLE_MAX_INDICATORS} (OR signals)")
        print(
            f"Boruta: IS Sharpe > {ENSEMBLE_BORUTA_MIN_IS_SHARPE}, "
            f"{ENSEMBLE_BORUTA_SHUFFLES} OOS shuffles"
        )
        print("=" * 70)

        for _, row in full_rows.iterrows():
            indicator_name = str(row["indicator"])

            try:
                params = _params_from_full_row(indicator_name, row)
                spec = get_active_indicator_spec(indicator_name)
                if "param_display" in row.index and pd.notna(row.get("param_display")):
                    param_display = str(row.get("param_display"))
                else:
                    param_display = spec["format_params"](params)
            except Exception as exc:
                print(f"  SKIP {indicator_name}: could not rebuild params from CSV ({exc})")
                continue

            csv_is_sharpe = _finite_sharpe_value(row.get("is_sharpe")) if "is_sharpe" in full_rows.columns else np.nan
            if not np.isfinite(csv_is_sharpe):
                csv_is_sharpe = _sharpe_matching_params(
                    is_rows, indicator_name, params, ("sharpe_ratio", "sharpe", "IS_Sharpe")
                )
            if not np.isfinite(csv_is_sharpe):
                csv_is_sharpe = _lookup_stage_sharpe(is_sharpe_by_indicator, indicator_name)

            csv_oos_sharpe = _finite_sharpe_value(row.get("oos_sharpe")) if "oos_sharpe" in full_rows.columns else np.nan
            matched_oos = False
            if np.isfinite(csv_oos_sharpe):
                matched_oos = True
            else:
                csv_oos_sharpe = _sharpe_matching_params(
                    oos_rows, indicator_name, params, ("OOS_Sharpe", "sharpe", "oos_sharpe")
                )
                if np.isfinite(csv_oos_sharpe):
                    matched_oos = True
            used_oos_max_fallback = False
            if not np.isfinite(csv_oos_sharpe):
                csv_oos_sharpe = _lookup_stage_sharpe(oos_sharpe_by_indicator, indicator_name)
                used_oos_max_fallback = np.isfinite(csv_oos_sharpe)

            is_sharpe, oos_sharpe, sharpe_source, oos_from_csv = _resolve_pass_sharpes(
                indicator_name, csv_is_sharpe, csv_oos_sharpe, csv_params=params
            )
            if oos_from_csv:
                oos_gate_used_csv = True

            selection_source = ""
            if "selection_source" in full_rows.columns:
                selection_source = str(row.get("selection_source", "")).strip().lower()

            passes_oos = np.isfinite(oos_sharpe) and oos_sharpe > ENSEMBLE_MIN_OOS_SHARPE
            status = "PASS" if passes_oos else "FAIL"
            fail_note = ""
            extra_notes = []
            if status == "FAIL":
                if not np.isfinite(oos_sharpe):
                    extra_notes.append("missing/non-finite OOS (oos stage row?)")
                elif not passes_oos:
                    extra_notes.append(f"OOS <= {ENSEMBLE_MIN_OOS_SHARPE}")
            if (
                selection_source
                and selection_source not in {"oos", "n/a", "nan", "none", ""}
            ):
                extra_notes.append(f"full selection_source={selection_source!r} (not oos)")
            if extra_notes:
                fail_note = " [" + "; ".join(extra_notes) + "]"
            print(
                f"  {status:4s} {indicator_name:12s} {param_display:28s} "
                f"IS={fmt_metric(is_sharpe, '.3f')} OOS={fmt_metric(oos_sharpe, '.3f')} "
                f"({sharpe_source}){fail_note}"
            )
            if not passes_oos:
                continue
            if oos_from_csv and used_oos_max_fallback and not matched_oos:
                oos_pass_used_csv_fallback = True

            ensemble_candidates[indicator_name] = {
                "params": params,
                "param_display": param_display,
                "spec": spec,
                "is_sharpe": is_sharpe,
                "oos_sharpe": oos_sharpe,
                "sharpe_source": sharpe_source,
                "selection_source": selection_source or "n/a",
                "oos_from_csv": bool(oos_from_csv),
            }

        if oos_gate_used_csv:
            print(
                "Note: OOS values fell back to CSV for at least one indicator "
                "(full-row ref, param-matched OOS row, or top-N max)."
            )
        if oos_pass_used_csv_fallback:
            print(
                "Note: at least one PASS was gated on CSV top-N OOS MAX because the "
                "selected full-sample params were not found in stage='oos' rows. "
                "Prefer stage='oos' CSV rows that match the selected full-sample params."
            )

        eligible_names = list(ensemble_candidates.keys())
        print("-" * 70)
        print(f"Eligible indicators for ensembles: {len(eligible_names)}")
        if len(eligible_names) < ENSEMBLE_MIN_INDICATORS:
            raise ValueError(
                f"Need at least {ENSEMBLE_MIN_INDICATORS} eligible indicators for ensemble search; "
                f"got {len(eligible_names)}. Loosen ENSEMBLE_MIN_OOS_SHARPE."
            )

        # ---------------------------------------------------------------------------
        # Pre-compute per-indicator signals on IS only (OOS built later for threshold passers)
        # ---------------------------------------------------------------------------
        print("\nBuilding member signals on training (IS) data...")
        member_signals_is = {}

        for indicator_name, info in list(ensemble_candidates.items()):
            try:
                entries, exits = info["spec"]["build_signals"](
                    train_close, info["params"], shift_signals=True
                )
                if len(entries) != len(train_close) or len(exits) != len(train_close):
                    raise ValueError(
                        f"signal length mismatch "
                        f"(entries={len(entries)}, exits={len(exits)}, close={len(train_close)})"
                    )
                member_signals_is[indicator_name] = (
                    pd.Series(np.asarray(entries, dtype=bool), index=train_close.index, dtype=bool),
                    pd.Series(np.asarray(exits, dtype=bool), index=train_close.index, dtype=bool),
                )
            except Exception as exc:
                print(
                    f"  SKIP {indicator_name}: signal build failed on training (IS) "
                    f"({type(exc).__name__}: {exc})"
                )
                del ensemble_candidates[indicator_name]

        eligible_names = list(ensemble_candidates.keys())
        if len(eligible_names) < ENSEMBLE_MIN_INDICATORS:
            raise ValueError(
                f"After signal-build failures, need at least {ENSEMBLE_MIN_INDICATORS} "
                f"eligible indicators; got {len(eligible_names)}."
            )
        print(
            f"Cached IS signals for {len(member_signals_is)} indicators "
            f"({len(eligible_names)} eligible)."
        )

        # ---------------------------------------------------------------------------
        # OR ensemble grid search — IS backtests, then Boruta on IS Sharpe threshold
        # ---------------------------------------------------------------------------
        ensemble_grid = []
        for size in range(ENSEMBLE_MIN_INDICATORS, ENSEMBLE_MAX_INDICATORS + 1):
            ensemble_grid.extend(combinations(eligible_names, size))

        train_years = sample_years(train_close)
        val_years = sample_years(val_close)
        for label, years in (
            ("train_close", train_years),
            ("val_close", val_years),
        ):
            if not np.isfinite(years) or years <= 0:
                raise ValueError(f"sample_years({label}) must be > 0; got {years!r}")


        print("\nINITIATING ENSEMBLE GRID SEARCH (OR) + BORUTA")
        print("=" * 70)
        print(f"Training Period:    {train_close.index[0].date()} -> {train_close.index[-1].date()}")
        print(f"Validation Period:  {val_close.index[0].date()} -> {val_close.index[-1].date()}")
        print(f"Combinations to test: {len(ensemble_grid)}")
        print("Signal combine rule: OR (any entry / any exit)")
        print(
            f"Selection: IS Sharpe > {ENSEMBLE_BORUTA_MIN_IS_SHARPE}, "
            f"then Boruta ({ENSEMBLE_BORUTA_SHUFFLES} OOS shuffles, "
            f"seed={ENSEMBLE_BORUTA_RANDOM_SEED})"
        )
        print("Ranking: IS Sharpe (descending)")
        print("=" * 70)

        ensemble_search_results = []
        ensemble_failed = 0
        ensemble_failure_details = []

        for members in ensemble_grid:
            if not members:
                raise ValueError("Empty ensemble member list.")
            label = " | ".join(
                f"{name} {ensemble_candidates[name]['param_display']}" for name in members
            )
            try:
                is_metrics = _backtest_ensemble(train_close, member_signals_is, members, train_years)

                members = tuple(members)
                ensemble_search_results.append({
                    "ensemble": " OR ".join(members),
                    "members": members,
                    "n_members": len(members),
                    "param_display": label,
                    "combine_rule": "OR",
                    "is_sharpe": is_metrics["sharpe"],
                    "is_sortino": is_metrics["sortino"],
                    "is_total_return": is_metrics["total_return"],
                    "is_max_drawdown": is_metrics["max_drawdown"],
                    "is_total_trades": is_metrics["total_trades"],
                    "is_win_rate": is_metrics["win_rate"],
                    "is_profit_factor": is_metrics["profit_factor"],
                    "is_trades_per_year": is_metrics["trades_per_year"],
                })
            except Exception as exc:
                ensemble_failed += 1
                if len(ensemble_failure_details) < 10:
                    ensemble_failure_details.append(f"{label}: {type(exc).__name__}: {exc}")

        ensemble_is_df = pd.DataFrame(ensemble_search_results)
        if ensemble_is_df.empty:
            raise ValueError("Ensemble grid search produced no stored results.")

        print("\nENSEMBLE GRID SEARCH COMPLETED!")
        print("=" * 70)
        print(f"Stored IS ensembles: {len(ensemble_is_df)}")
        print(f"Failed ensembles: {ensemble_failed}")
        if ensemble_failure_details:
            print("Sample failures:")
            for detail in ensemble_failure_details:
                print(f"  - {detail}")

        ensemble_is_df = ensemble_is_df.dropna(subset=["is_sharpe"]).reset_index(drop=True)
        if ensemble_is_df.empty:
            raise ValueError("Ensemble grid search produced no finite IS Sharpe results.")

        _is_sharpe_ok = np.isfinite(ensemble_is_df["is_sharpe"]) & (
            ensemble_is_df["is_sharpe"] > ENSEMBLE_BORUTA_MIN_IS_SHARPE
        )
        oos_candidate_ensembles = (
            ensemble_is_df.loc[_is_sharpe_ok]
            .sort_values("is_sharpe", ascending=False, na_position="last")
            .reset_index(drop=True)
        )
        if oos_candidate_ensembles.empty:
            _best_is = safe_float(ensemble_is_df["is_sharpe"].max())
            raise ValueError(
                f"No ensembles with IS Sharpe > {ENSEMBLE_BORUTA_MIN_IS_SHARPE} for Boruta validation; "
                f"best IS Sharpe={fmt_metric(_best_is, '.3f')} across {len(ensemble_is_df)}. "
                f"Loosen ENSEMBLE_BORUTA_MIN_IS_SHARPE."
            )
        selection_label = (
            f"IS Sharpe > {ENSEMBLE_BORUTA_MIN_IS_SHARPE} "
            f"({len(oos_candidate_ensembles)} ensembles)"
        )

        print(f"\nIN-SAMPLE ENSEMBLES — BORUTA / OOS VALIDATION")
        print("=" * 70)
        print(f"Candidates: {selection_label}")
        print("=" * 70)

        # Build OOS signals only for indicators that appear in the selected IS ensembles
        needed_oos_members = set()
        for raw_members in oos_candidate_ensembles["members"]:
            members = tuple(raw_members)
            if not members:
                raise ValueError("Empty ensemble member list in OOS-candidate IS results.")
            needed_oos_members.update(members)

        print(
            f"\nBuilding member signals on validation (OOS) data "
            f"for {len(needed_oos_members)} candidate member indicator(s)..."
        )
        member_signals_oos = {}
        for indicator_name in sorted(needed_oos_members):
            info = ensemble_candidates.get(indicator_name)
            if info is None:
                print(f"  SKIP {indicator_name}: missing from ensemble_candidates")
                continue
            try:
                entries, exits = info["spec"]["build_signals"](
                    val_close, info["params"], shift_signals=True
                )
                if len(entries) != len(val_close) or len(exits) != len(val_close):
                    raise ValueError(
                        f"signal length mismatch "
                        f"(entries={len(entries)}, exits={len(exits)}, close={len(val_close)})"
                    )
                member_signals_oos[indicator_name] = (
                    pd.Series(np.asarray(entries, dtype=bool), index=val_close.index, dtype=bool),
                    pd.Series(np.asarray(exits, dtype=bool), index=val_close.index, dtype=bool),
                )
            except Exception as exc:
                print(
                    f"  SKIP {indicator_name}: signal build failed on validation (OOS) "
                    f"({type(exc).__name__}: {exc})"
                )

        ensemble_oos_results = []
        ensemble_oos_failed = 0
        ensemble_oos_failure_details = []
        def _oos_boruta_sharpe_fn(entries, exits, _price=val_close):
            return safe_float(
                run_signal_backtest(_price, entries, exits).sharpe_ratio(
                    freq=FREQ, year_freq=YEAR_FREQ
                )
            )

        for is_rank, (_, row) in enumerate(oos_candidate_ensembles.iterrows(), 1):
            members = tuple(row["members"])
            label = row["param_display"]
            try:
                missing_oos = [name for name in members if name not in member_signals_oos]
                if missing_oos:
                    raise KeyError(
                        f"missing OOS signals for member(s): {missing_oos}"
                    )
                oos_metrics = _backtest_ensemble(
                    val_close, member_signals_oos, members, val_years, need_returns=False
                )
                boruta_score = _boruta_score_from_signals(
                    {name: member_signals_oos[name] for name in members},
                    _oos_boruta_sharpe_fn,
                    ENSEMBLE_BORUTA_SHUFFLES,
                    ENSEMBLE_BORUTA_RANDOM_SEED,
                )
                ensemble_oos_results.append({
                    "ensemble": row["ensemble"],
                    "members": members,
                    "n_members": row["n_members"],
                    "param_display": label,
                    "combine_rule": row["combine_rule"],
                    "is_rank": is_rank,
                    "is_sharpe": row["is_sharpe"],
                    "oos_sharpe": oos_metrics["sharpe"],
                    "boruta_score": boruta_score,
                    "is_sortino": row["is_sortino"],
                    "oos_sortino": oos_metrics["sortino"],
                    "is_total_return": row["is_total_return"],
                    "oos_total_return": oos_metrics["total_return"],
                    "is_max_drawdown": row["is_max_drawdown"],
                    "oos_max_drawdown": oos_metrics["max_drawdown"],
                    "is_total_trades": row["is_total_trades"],
                    "oos_total_trades": oos_metrics["total_trades"],
                    "is_win_rate": row["is_win_rate"],
                    "oos_win_rate": oos_metrics["win_rate"],
                    "is_profit_factor": row["is_profit_factor"],
                    "oos_profit_factor": oos_metrics["profit_factor"],
                    "is_trades_per_year": row["is_trades_per_year"],
                    "oos_trades_per_year": oos_metrics["trades_per_year"],
                })
            except Exception as exc:
                ensemble_oos_failed += 1
                if len(ensemble_oos_failure_details) < 10:
                    ensemble_oos_failure_details.append(
                        f"IS#{is_rank} {label}: {type(exc).__name__}: {exc}"
                    )

        ensemble_results_df = pd.DataFrame(ensemble_oos_results)
        if ensemble_results_df.empty:
            raise ValueError(
                f"Boruta/OOS validation of {len(oos_candidate_ensembles)} IS ensembles "
                "produced no stored results."
            )

        # Optimize / rank by in-sample Sharpe (Boruta score is a validation metric).
        ensemble_results_df = (
            ensemble_results_df
            .sort_values("is_sharpe", ascending=False, na_position="last")
            .reset_index(drop=True)
        )

        print(f"Boruta/OOS-validated ensembles: {len(ensemble_results_df)}")
        print(f"Failed Boruta/OOS ensembles: {ensemble_oos_failed}")
        _boruta_ok = int(np.isfinite(ensemble_results_df["boruta_score"]).sum())
        _boruta_nan = int(len(ensemble_results_df) - _boruta_ok)
        print(
            f"Boruta scores: {_boruta_ok} finite, {_boruta_nan} n/a "
            f"(need >= {ENSEMBLE_BORUTA_MIN_OOS_BARS} OOS bars and std > 0)"
        )
        if _boruta_ok == 0:
            print(
                "WARNING: No finite Boruta scores were produced; "
                "ranking is IS Sharpe only."
            )
        if ensemble_oos_failure_details:
            print("Sample Boruta/OOS failures:")
            for detail in ensemble_oos_failure_details:
                print(f"  - {detail}")

        ensemble_results_df = ensemble_results_df.assign(ticker=TICKER)
        _cols = ["ticker"] + [c for c in ensemble_results_df.columns if c != "ticker"]
        ensemble_results_df = ensemble_results_df.loc[:, _cols]
        ensemble_results_by_ticker[TICKER] = ensemble_results_df
        all_ensemble_frames.append(ensemble_results_df)
        print(f"Stored {len(ensemble_results_df)} Boruta/OOS ensembles for {TICKER}")

    except Exception as exc:
        _ensemble_ticker_failures.append(f"{TICKER}: {type(exc).__name__}: {exc}")
        print(f"SKIP {TICKER}: {type(exc).__name__}: {exc}")
        continue

# Restore active ticker / in-memory bests for any downstream cells
TICKER = _original_ticker
if _saved_best_by_indicator is not None:
    best_by_indicator = _saved_best_by_indicator

# ---------------------------------------------------------------------------
# Master leaderboard + per-ticker showcase tables
# ---------------------------------------------------------------------------
try:
    from IPython.display import HTML as _notebook_HTML
    from IPython.display import display as _notebook_display
except ImportError:  # plain Python / missing IPython
    _notebook_HTML = None
    _notebook_display = None

# Pandas truncates HTML/text repr around this many rows (head/tail). Scroll above it.
_SHOW_DF_SCROLL_ROWS = 60


def _show_dataframe(frame):
    """Render a DataFrame as a notebook table when possible; else print."""
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(
            f"_show_dataframe expected a pandas DataFrame, got {type(frame).__name__}"
        )

    if _notebook_display is not None:
        if _notebook_HTML is not None and len(frame.index) > _SHOW_DF_SCROLL_ROWS:
            _notebook_display(
                _notebook_HTML(
                    '<div style="max-height:420px;overflow:auto">'
                    + frame.to_html(escape=True)
                    + "</div>"
                )
            )
        else:
            with pd.option_context("display.max_rows", None):
                _notebook_display(frame)
        return

    # Fallback for non-notebook runners (nbclient still has IPython; scripts may not).
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
        print(frame.to_string())


def _fmt_frac_pct(value):
    number = safe_float(value)
    if not np.isfinite(number):
        return "n/a"
    return f"{number:.2%}"


def _fmt_win_pct(value):
    # trade_stats stores win rate in percentage points (0-100), not 0-1 fractions.
    number = safe_float(value)
    if not np.isfinite(number):
        return "n/a"
    return f"{number:.2f}%"


def _fmt_boruta_score(value):
    number = safe_float(value)
    if not np.isfinite(number):
        return np.nan
    return int(round(number))


def _fmt_trades(value):
    number = safe_float(value)
    if not np.isfinite(number):
        return np.nan
    return int(number)


def _fmt_profit_factor(value):
    # Do not route through safe_float: it collapses +/-inf (zero-loss books) to NaN.
    try:
        number = float(value)
    except (TypeError, ValueError):
        return np.nan
    if np.isnan(number):
        return np.nan
    if np.isposinf(number):
        return np.inf
    if np.isneginf(number):
        return -np.inf
    return number


def _build_showcase_df(frame):
    """Build the detailed ensemble showcase table for one ticker's results."""
    _showcase_required = {
        "ensemble",
        "is_sharpe",
        "oos_sharpe",
        "boruta_score",
        "oos_total_return",
        "oos_win_rate",
        "oos_profit_factor",
        "oos_max_drawdown",
        "oos_total_trades",
        "param_display",
    }
    _missing_showcase = _showcase_required.difference(frame.columns)
    if _missing_showcase:
        raise KeyError(
            "ensemble results missing columns required for showcase table: "
            f"{sorted(_missing_showcase)}"
        )
    return pd.DataFrame({
        "ensemble": frame["ensemble"].astype(str),
        "IS_Sharpe": frame["is_sharpe"].map(safe_float),
        "OOS_Sharpe": frame["oos_sharpe"].map(safe_float),
        "Boruta_Score": frame["boruta_score"].map(_fmt_boruta_score),
        "Total_Return": frame["oos_total_return"].map(_fmt_frac_pct),
        "WinRate": frame["oos_win_rate"].map(_fmt_win_pct),
        "ProfitFactor": frame["oos_profit_factor"].map(_fmt_profit_factor),
        "MaxDD": frame["oos_max_drawdown"].map(_fmt_frac_pct),
        "Num_Trades": frame["oos_total_trades"].map(_fmt_trades),
    })


if _ensemble_ticker_failures:
    print("\n" + "=" * 120)
    print(f"TICKER FAILURES: {len(_ensemble_ticker_failures)}")
    print("=" * 120)
    for detail in _ensemble_ticker_failures:
        print(f"  - {detail}")

if not all_ensemble_frames:
    raise ValueError(
        "Ensemble grid search produced no stored results for any ticker in "
        f"{_ensemble_tickers!r}."
    )

ensemble_results_df = (
    pd.concat(all_ensemble_frames, ignore_index=True)
    .sort_values(["ticker", "is_sharpe"], ascending=[True, False], na_position="last")
    .reset_index(drop=True)
)

# A. MASTER LEADERBOARD: best ensemble per ticker
# Highest IS Sharpe with OOS Sharpe >= ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE
# (walk down IS ranks until the gate clears; else highest IS Sharpe + WARNING)
_lb_required = {"ticker", "is_sharpe", "oos_sharpe"}
_lb_missing = _lb_required.difference(ensemble_results_df.columns)
if _lb_missing:
    raise KeyError(
        "ensemble results missing columns required for leaderboard: "
        f"{sorted(_lb_missing)}"
    )
if ensemble_results_df.empty:
    raise ValueError("ensemble_results_df is empty; cannot build leaderboard.")

_is_vals = ensemble_results_df["is_sharpe"].map(safe_float)
_oos_vals = ensemble_results_df["oos_sharpe"].map(safe_float)
_oos_ok = np.isfinite(_oos_vals) & (_oos_vals >= ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE)
_picked = (
    ensemble_results_df
    .assign(_oos_gate=_oos_ok, _is_sharpe=_is_vals)
    .sort_values(["_oos_gate", "_is_sharpe"], ascending=[False, False], na_position="last")
    .drop_duplicates(subset=["ticker"], keep="first")
)
_fallback_tickers = (
    _picked.loc[~_picked["_oos_gate"], "ticker"].astype(str).tolist()
)
if _fallback_tickers:
    print(
        f"\nWARNING: No ensemble with OOS Sharpe >= {ENSEMBLE_LEADERBOARD_MIN_OOS_SHARPE} "
        f"for {len(_fallback_tickers)} ticker(s); using highest IS Sharpe instead: "
        f"{', '.join(_fallback_tickers)}"
    )
leaderboard_df = (
    _picked
    .drop(columns=["_oos_gate", "_is_sharpe"])
    .sort_values("is_sharpe", ascending=False, na_position="last")
    .reset_index(drop=True)
)
if leaderboard_df.empty:
    raise ValueError("Leaderboard selection produced no rows.")

leaderboard_showcase_df = pd.DataFrame({
    "ticker": leaderboard_df["ticker"].astype(str),
    "ensemble": leaderboard_df["ensemble"].astype(str),
    "IS_Sharpe": leaderboard_df["is_sharpe"].map(safe_float),
    "OOS_Sharpe": leaderboard_df["oos_sharpe"].map(safe_float),
    "Boruta_Score": leaderboard_df["boruta_score"].map(_fmt_boruta_score),
    "Total_Return": leaderboard_df["oos_total_return"].map(_fmt_frac_pct),
    "WinRate": leaderboard_df["oos_win_rate"].map(_fmt_win_pct),
    "ProfitFactor": leaderboard_df["oos_profit_factor"].map(_fmt_profit_factor),
    "MaxDD": leaderboard_df["oos_max_drawdown"].map(_fmt_frac_pct),
    "Num_Trades": leaderboard_df["oos_total_trades"].map(_fmt_trades),
})

print("\n" + "=" * 120)
print(
    f"MASTER LEADERBOARD: BEST ENSEMBLE FOR EACH OF THE "
    f"{len(leaderboard_showcase_df)} ASSETS"
)
print("=" * 120)
_show_dataframe(leaderboard_showcase_df)

# B. DETAILED SHOWCASE: full ensemble list per ticker (same layout as before)
print("\n" + "=" * 120)
print("DETAILED SHOWCASE: ENSEMBLE COMBINATIONS PER ASSET (Ranked by IS Sharpe)")
print("=" * 120)

showcase_by_ticker = {}
for _ticker_label in _ensemble_tickers:
    if _ticker_label not in ensemble_results_by_ticker:
        continue
    _ticker_frame = ensemble_results_by_ticker[_ticker_label]
    _n_show = len(_ticker_frame)
    showcase_df = _build_showcase_df(_ticker_frame)
    showcase_by_ticker[_ticker_label] = showcase_df
    print(f"\n{_ticker_label}:")
    print("-" * 120)
    print(f"TOP {_n_show} ENSEMBLE COMBINATIONS (Ranked by IS Sharpe)")
    _show_dataframe(showcase_df)

# Convenience aliases point at the leaderboard-best ticker (highest IS Sharpe)
_best_ticker = str(leaderboard_df.iloc[0]["ticker"])
if _best_ticker not in showcase_by_ticker:
    raise KeyError(
        f"Leaderboard ticker {_best_ticker!r} missing from showcase_by_ticker "
        f"(have {sorted(showcase_by_ticker)})."
    )
showcase_df = showcase_by_ticker[_best_ticker]
best_ensemble = leaderboard_df.iloc[0]

best_is_sharpe = safe_float(best_ensemble["is_sharpe"])
best_boruta = safe_float(best_ensemble["boruta_score"])
if not np.isfinite(best_is_sharpe):
    print(
        "\nWARNING: Best ensemble IS Sharpe is non-finite; ranking is informational only."
    )
if not np.isfinite(best_boruta):
    print(
        "\nWARNING: Best ensemble Boruta Score is n/a "
        "(insufficient/degenerate OOS returns)."
    )

print("\n" + "-" * 120)
print(f"LEADERBOARD BEST: {leaderboard_df.iloc[0]['ticker']} | {leaderboard_df.iloc[0]['ensemble']}")
print(f"IS Sharpe:     {fmt_metric(leaderboard_df.iloc[0]['is_sharpe'], '.3f')}")
print(f"OOS Sharpe:    {fmt_metric(leaderboard_df.iloc[0]['oos_sharpe'], '.3f')}")
print(f"Boruta Score:  {fmt_metric(leaderboard_df.iloc[0]['boruta_score'], '.1f')}")
print(f"Members:       {leaderboard_df.iloc[0]['param_display']}")
print(
    "Results stored in 'ensemble_results_df' "
    "(leaderboard: 'leaderboard_showcase_df'; per-ticker: 'showcase_by_ticker')"
)


ENROLLING ENSEMBLE SEARCH FOR 11 TICKERS
Tickers: BTC-USD, ETH-USD, TQQQ, UPRO, SMH.L, UCO, AAPL, GOOG, GLD, QQQ, SPY

########################################################################################################################
# TICKER: BTC-USD
########################################################################################################################
ENSEMBLE ELIGIBILITY (from results CSV full-sample params)
Results file: C:\Quant Rick's Trading Academy\Indicator_sweep_results\btc_usd_indicator_sweep_results.csv
Pass threshold: OOS Sharpe > 0.7
Ensemble sizes: 2 to 3 (OR signals)
Boruta: IS Sharpe > 0.95, 50 OOS shuffles
  PASS Triple EMA   EMA(5,104,121)               IS=1.259 OOS=0.729 (csv)
  PASS MACD         MACD(40,69,89)               IS=1.478 OOS=0.973 (csv)
  PASS AROON        AROON(16)                    IS=1.126 OOS=1.158 (csv)
  PASS STC          STC(39,65,35)                IS=1.190 OOS=1.061 (csv)
  FAIL KAMA         KAMA(5,7,71)                 

Cached IS signals for 11 indicators (11 eligible).

INITIATING ENSEMBLE GRID SEARCH (OR) + BORUTA
Training Period:    2018-01-01 -> 2023-03-01
Validation Period:  2023-03-02 -> 2026-08-10
Combinations to test: 220
Signal combine rule: OR (any entry / any exit)
Selection: IS Sharpe > 0.95, then Boruta (50 OOS shuffles, seed=42)
Ranking: IS Sharpe (descending)



ENSEMBLE GRID SEARCH COMPLETED!
Stored IS ensembles: 220
Failed ensembles: 0

IN-SAMPLE ENSEMBLES — BORUTA / OOS VALIDATION
Candidates: IS Sharpe > 0.95 (218 ensembles)

Building member signals on validation (OOS) data for 11 candidate member indicator(s)...


Boruta/OOS-validated ensembles: 218
Failed Boruta/OOS ensembles: 0
Boruta scores: 218 finite, 0 n/a (need >= 11 OOS bars and std > 0)
Stored 218 Boruta/OOS ensembles for BTC-USD

########################################################################################################################
# TICKER: ETH-USD
########################################################################################################################
ENSEMBLE ELIGIBILITY (from results CSV full-sample params)
Results file: C:\Quant Rick's Trading Academy\Indicator_sweep_results\eth_usd_indicator_sweep_results.csv
Pass threshold: OOS Sharpe > 0.7
Ensemble sizes: 2 to 3 (OR signals)
Boruta: IS Sharpe > 0.95, 50 OOS shuffles
  PASS Triple EMA   EMA(7,95,132)                IS=1.098 OOS=0.706 (csv)
  PASS MACD         MACD(19,65,40)               IS=1.208 OOS=1.005 (csv)
  FAIL AROON        AROON(18)                    IS=1.356 OOS=0.238 (csv) [OOS <= 0.7]
  PASS STC          STC(34,75,21)                I


ENSEMBLE GRID SEARCH COMPLETED!
Stored IS ensembles: 20
Failed ensembles: 0

IN-SAMPLE ENSEMBLES — BORUTA / OOS VALIDATION
Candidates: IS Sharpe > 0.95 (20 ensembles)

Building member signals on validation (OOS) data for 5 candidate member indicator(s)...


...


QQQ:
------------------------------------------------------------------------------------------------------------------------
TOP 123 ENSEMBLE COMBINATIONS (Ranked by IS Sharpe)


,ensemble,IS_Sharpe,OOS_Sharpe,Boruta_Score,Total_Return,WinRate,ProfitFactor,MaxDD,Num_Trades
0,Triple EMA OR MACD OR STC,1.469207,1.307067,4,67.36%,50.00%,4.896361,-11.83%,18
1,Triple EMA OR STC OR ALMA,1.382457,1.044625,47,51.92%,52.00%,3.097686,-9.95%,25
2,Triple EMA OR MACD OR KAMA,1.343667,1.335431,22,83.86%,50.00%,4.240423,-12.94%,22
3,Triple EMA OR STC OR VORTEX,1.336709,1.354446,0,73.83%,62.50%,4.614039,-9.07%,24
4,Triple EMA OR STC,1.334432,1.310545,33,69.08%,50.00%,4.612913,-11.83%,18
5,ADX OR VORTEX OR ALMA,1.331018,1.564780,35,96.16%,63.33%,6.458341,-10.83%,30
6,Triple EMA OR MACD OR DONCHIAN,1.320495,1.503153,3,89.53%,52.94%,5.718966,-10.89%,17
7,MACD OR STC OR RSI,1.301412,1.424149,1,74.65%,64.29%,5.698495,-9.41%,14
8,Triple EMA OR STC OR RSI,1.296828,1.305401,97,69.34%,57.14%,4.612106,-9.41%,21
9,MACD OR STC OR VORTEX,1.294862,1.458871,72,77.58%,70.59%,5.654781,-9.07%,17



SPY:
------------------------------------------------------------------------------------------------------------------------
TOP 32 ENSEMBLE COMBINATIONS (Ranked by IS Sharpe)


,ensemble,IS_Sharpe,OOS_Sharpe,Boruta_Score,Total_Return,WinRate,ProfitFactor,MaxDD,Num_Trades
0,STC OR VORTEX OR ALMA,1.324340,1.859668,0,72.59%,64.71%,10.211811,-5.08%,17
1,STC OR ALMA,1.204889,1.614325,47,53.74%,66.67%,9.239970,-6.94%,12
2,Triple EMA OR STC OR ALMA,1.155026,1.552349,4,54.32%,61.54%,6.787288,-6.94%,13
3,MACD OR STC OR ALMA,1.154387,1.278448,0,42.10%,55.56%,4.601108,-8.02%,18
4,STC OR ADX OR ALMA,1.152683,1.040985,0,34.06%,47.06%,3.100787,-8.63%,17
5,Triple EMA OR STC OR VORTEX,1.114440,1.769471,40,68.74%,57.89%,7.166444,-6.49%,19
6,STC OR VORTEX,1.092742,1.888805,76,74.20%,58.82%,10.032369,-5.08%,17
7,Triple EMA OR STC,1.074616,1.591997,36,56.17%,58.33%,6.997348,-6.94%,12
8,ADX OR VORTEX OR ALMA,1.046216,1.054818,0,37.67%,55.00%,3.037726,-7.87%,20
9,MACD OR RSI OR TRIX,1.024091,1.438203,18,56.53%,50.00%,3.109487,-10.73%,32



------------------------------------------------------------------------------------------------------------------------
LEADERBOARD BEST: ETH-USD | Triple EMA OR MACD OR STC
IS Sharpe:     1.563
OOS Sharpe:    1.098
Boruta Score:  3.3
Members:       Triple EMA EMA(7,95,132) | MACD MACD(19,65,40) | STC STC(34,75,21)
Results stored in 'ensemble_results_df' (leaderboard: 'leaderboard_showcase_df'; per-ticker: 'showcase_by_ticker')
